<a href="https://colab.research.google.com/github/amzad-786githumb/SPP_GAN_Research/blob/main/19_Statistical_Significance_Testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
# ============================================================
# NOTEBOOK 19 — STATISTICAL SIGNIFICANCE TESTING
# SECTION 1 — HEADER & SCOPE
# ============================================================

# ------------------------------------------------------------
# 1. GOOGLE DRIVE AND IMPORTS
# ------------------------------------------------------------

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import gc
import json
import hashlib
import warnings
import itertools

from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from scipy import stats

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 2. PROJECT CONFIGURATION
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/SPP_GAN_Research"
)

NOTEBOOK_ID = "notebook_19"

NOTEBOOK_NAME = "Statistical Significance Testing"

NOTEBOOK_ROOT = (
    PROJECT_ROOT
    / "results"
    / "notebooks"
    / NOTEBOOK_ID
)

RESULTS_DIR = NOTEBOOK_ROOT / "results"
TABLES_DIR = NOTEBOOK_ROOT / "tables"
FIGURES_DIR = NOTEBOOK_ROOT / "figures"
METADATA_DIR = NOTEBOOK_ROOT / "metadata"
VALIDATION_DIR = NOTEBOOK_ROOT / "validation"
MANIFESTS_DIR = NOTEBOOK_ROOT / "manifests"

DIRECTORIES = {
    "results": RESULTS_DIR,
    "tables": TABLES_DIR,
    "figures": FIGURES_DIR,
    "metadata": METADATA_DIR,
    "validation": VALIDATION_DIR,
    "manifests": MANIFESTS_DIR
}

for directory in DIRECTORIES.values():
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

# ------------------------------------------------------------
# 3. EXPERIMENTAL DESIGN
# ------------------------------------------------------------

MASTER_SEED = 2025

DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us"
]

VARIANTS = [
    "gan_only",
    "statistical_guidance_gan",
    "gan_dp",
    "full_sppgan"
]

DP_VARIANTS = [
    "gan_dp",
    "full_sppgan"
]

REPETITIONS = [1, 2, 3, 4, 5]

REPETITION_SEEDS = {
    repetition: MASTER_SEED + 1000 + repetition
    for repetition in REPETITIONS
}

KEY_COLUMNS = [
    "dataset",
    "variant",
    "repetition"
]

EXPECTED_RECORDS = (
    len(DATASETS)
    * len(VARIANTS)
    * len(REPETITIONS)
)

EXPECTED_RECORDS_PER_DATASET = (
    len(VARIANTS)
    * len(REPETITIONS)
)

EXPECTED_RECORDS_PER_VARIANT = (
    len(DATASETS)
    * len(REPETITIONS)
)

EXPECTED_DP_RECORDS = (
    len(DATASETS)
    * len(DP_VARIANTS)
    * len(REPETITIONS)
)

EXPECTED_NON_DP_RECORDS = (
    EXPECTED_RECORDS - EXPECTED_DP_RECORDS
)

# ------------------------------------------------------------
# 4. STATISTICAL TESTING CONFIGURATION
# ------------------------------------------------------------

ALPHA = 0.05

CONFIDENCE_LEVEL = 0.95

BOOTSTRAP_ITERATIONS = 10000

STATISTICAL_SEED = MASTER_SEED + 1900

MULTIPLE_TEST_METHOD = "holm"

MINIMUM_PAIRED_REPETITIONS = 3

# ------------------------------------------------------------
# 5. METRIC REGISTRY
# ------------------------------------------------------------
# Primary metrics must be derived from actual evaluated
# experimental results.
#
# privacy_score must represent verified formal DP protection.
# Empirical attack metrics remain separate.

PRIMARY_METRICS = [
    "fidelity_score",
    "utility_score",
    "privacy_score"
]

EMPIRICAL_PRIVACY_METRICS = [
    "membership_inference_auc",
    "mia_auc",
    "reidentification_risk",
    "disclosure_risk",
    "nearest_neighbor_disclosure_risk",
    "record_linkage_risk"
]

ALL_TEST_METRICS = (
    PRIMARY_METRICS
    + EMPIRICAL_PRIVACY_METRICS
)

# ------------------------------------------------------------
# 6. STATISTICAL PROTOCOL
# ------------------------------------------------------------

STATISTICAL_PROTOCOL = {

    "experimental_design": (
        "3 datasets x 4 variants x 5 repetitions"
    ),

    "experimental_unit": (
        "Independent experimental repetition"
    ),

    "blocking_variables": [
        "dataset",
        "repetition"
    ],

    "paired_test": (
        "Two-sided Wilcoxon signed-rank test"
    ),

    "omnibus_test": (
        "Friedman test"
    ),

    "posthoc_test": (
        "Pairwise Wilcoxon signed-rank tests"
    ),

    "effect_size": (
        "Paired rank-biserial correlation"
    ),

    "confidence_interval": (
        "Paired bootstrap percentile interval"
    ),

    "multiple_comparison_correction": (
        MULTIPLE_TEST_METHOD
    ),

    "significance_level": ALPHA,

    "confidence_level": CONFIDENCE_LEVEL,

    "bootstrap_iterations": BOOTSTRAP_ITERATIONS,

    "minimum_paired_repetitions": (
        MINIMUM_PAIRED_REPETITIONS
    ),

    "missing_data_policy": (
        "No imputation; incomplete pairs excluded "
        "and reported"
    ),

    "primary_metrics": PRIMARY_METRICS,

    "empirical_privacy_metrics": (
        EMPIRICAL_PRIVACY_METRICS
    ),

    "interpretation": (
        "Report effect sizes and uncertainty alongside "
        "p-values. Non-significance does not establish "
        "equivalence."
    )

}

# ------------------------------------------------------------
# 7. REPRODUCIBILITY
# ------------------------------------------------------------

np.random.seed(STATISTICAL_SEED)

RUN_TIMESTAMP = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%SZ")

# ------------------------------------------------------------
# 8. CONFIGURATION SNAPSHOT
# ------------------------------------------------------------

CONFIGURATION = {

    "project_root": str(PROJECT_ROOT),

    "notebook_id": NOTEBOOK_ID,

    "notebook_name": NOTEBOOK_NAME,

    "notebook_root": str(NOTEBOOK_ROOT),

    "master_seed": MASTER_SEED,

    "statistical_seed": STATISTICAL_SEED,

    "datasets": DATASETS,

    "variants": VARIANTS,

    "dp_variants": DP_VARIANTS,

    "repetitions": REPETITIONS,

    "repetition_seeds": {
        str(k): v
        for k, v in REPETITION_SEEDS.items()
    },

    "key_columns": KEY_COLUMNS,

    "expected_records": EXPECTED_RECORDS,

    "expected_dp_records": EXPECTED_DP_RECORDS,

    "expected_non_dp_records": EXPECTED_NON_DP_RECORDS,

    "alpha": ALPHA,

    "confidence_level": CONFIDENCE_LEVEL,

    "bootstrap_iterations": BOOTSTRAP_ITERATIONS,

    "minimum_paired_repetitions": (
        MINIMUM_PAIRED_REPETITIONS
    ),

    "multiple_test_method": MULTIPLE_TEST_METHOD,

    "primary_metrics": PRIMARY_METRICS,

    "empirical_privacy_metrics": (
        EMPIRICAL_PRIVACY_METRICS
    ),

    "statistical_protocol": STATISTICAL_PROTOCOL,

    "run_timestamp_utc": RUN_TIMESTAMP

}

CONFIG_PATH = (
    METADATA_DIR / "notebook_19_configuration.json"
)

with open(CONFIG_PATH, "w") as file:

    json.dump(
        CONFIGURATION,
        file,
        indent=2
    )

# ------------------------------------------------------------
# 9. SECTION 1 VALIDATION
# ------------------------------------------------------------

VALIDATION_CHECKS = [

    {
        "check": "project_root_exists",
        "passed": PROJECT_ROOT.exists()
    },

    {
        "check": "notebook_root_exists",
        "passed": NOTEBOOK_ROOT.exists()
    },

    {
        "check": "all_output_directories_created",
        "passed": all(
            directory.exists()
            for directory in DIRECTORIES.values()
        )
    },

    {
        "check": "expected_experimental_design",
        "passed": EXPECTED_RECORDS == 60
    },

    {
        "check": "expected_dp_design",
        "passed": EXPECTED_DP_RECORDS == 30
    },

    {
        "check": "repetition_seed_registry",
        "passed": (
            len(REPETITION_SEEDS) == len(REPETITIONS)
            and len(set(REPETITION_SEEDS.values()))
            == len(REPETITIONS)
        )
    },

    {
        "check": "statistical_configuration",
        "passed": (
            0 < ALPHA < 1
            and 0 < CONFIDENCE_LEVEL < 1
            and BOOTSTRAP_ITERATIONS >= 1000
            and MINIMUM_PAIRED_REPETITIONS >= 3
        )
    },

    {
        "check": "metric_registry",
        "passed": (
            len(ALL_TEST_METRICS)
            == len(set(ALL_TEST_METRICS))
        )
    },

    {
        "check": "configuration_saved",
        "passed": CONFIG_PATH.is_file()
    }

]

section_1_validation_df = pd.DataFrame(
    VALIDATION_CHECKS
)

SECTION_1_VALIDATION_PATH = (
    VALIDATION_DIR / "section_01_validation.csv"
)

section_1_validation_df.to_csv(
    SECTION_1_VALIDATION_PATH,
    index=False
)

SECTION_1_PASSED = int(
    section_1_validation_df["passed"].sum()
)

SECTION_1_FAILED = (
    len(section_1_validation_df)
    - SECTION_1_PASSED
)

SECTION_1_STATUS = (
    "PASS"
    if SECTION_1_FAILED == 0
    else "FAILED"
)

# ------------------------------------------------------------
# 10. SAVE SECTION 1 EXECUTION SUMMARY
# ------------------------------------------------------------

SECTION_1_SUMMARY = {

    "section": "SECTION 1 — HEADER & SCOPE",

    "status": SECTION_1_STATUS,

    "project_root": str(PROJECT_ROOT),

    "notebook_root": str(NOTEBOOK_ROOT),

    "expected_experiments": EXPECTED_RECORDS,

    "expected_dp_experiments": EXPECTED_DP_RECORDS,

    "datasets": len(DATASETS),

    "variants": len(VARIANTS),

    "repetitions": len(REPETITIONS),

    "validation_passed": SECTION_1_PASSED,

    "validation_failed": SECTION_1_FAILED,

    "configuration_path": str(CONFIG_PATH),

    "validation_path": str(
        SECTION_1_VALIDATION_PATH
    ),

    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat()

}

SECTION_1_SUMMARY_PATH = (
    METADATA_DIR / "section_01_summary.json"
)

with open(SECTION_1_SUMMARY_PATH, "w") as file:

    json.dump(
        SECTION_1_SUMMARY,
        file,
        indent=2
    )

# ------------------------------------------------------------
# 11. FINAL OUTPUT
# ------------------------------------------------------------

print("=" * 75)
print("NOTEBOOK 19 — SECTION 1: HEADER & SCOPE")
print("=" * 75)

print("Project root          :", PROJECT_ROOT)
print("Notebook root         :", NOTEBOOK_ROOT)

print("-" * 75)

print("Datasets              :", len(DATASETS))
print("Variants              :", len(VARIANTS))
print("Repetitions           :", len(REPETITIONS))

print("Expected experiments  :", EXPECTED_RECORDS)
print("Expected DP records   :", EXPECTED_DP_RECORDS)
print("Expected non-DP records:", EXPECTED_NON_DP_RECORDS)

print("-" * 75)

print("Significance level    :", ALPHA)
print("Confidence level      :", CONFIDENCE_LEVEL)
print("Bootstrap iterations  :", BOOTSTRAP_ITERATIONS)
print("Master seed           :", MASTER_SEED)
print("Statistical seed      :", STATISTICAL_SEED)

print("-" * 75)

print("Validation checks     :", len(VALIDATION_CHECKS))
print("Validation passed     :", SECTION_1_PASSED)
print("Validation failed     :", SECTION_1_FAILED)
print("Section 1 status      :", SECTION_1_STATUS)

print("-" * 75)

print("Configuration saved   :", CONFIG_PATH)
print("Validation saved      :", SECTION_1_VALIDATION_PATH)
print("Summary saved         :", SECTION_1_SUMMARY_PATH)

print("=" * 75)

assert SECTION_1_STATUS == "PASS", (
    "Section 1 validation failed. "
    "Review section_01_validation.csv."
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
NOTEBOOK 19 — SECTION 1: HEADER & SCOPE
Project root          : /content/drive/MyDrive/SPP_GAN_Research
Notebook root         : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19
---------------------------------------------------------------------------
Datasets              : 3
Variants              : 4
Repetitions           : 5
Expected experiments  : 60
Expected DP records   : 30
Expected non-DP records: 30
---------------------------------------------------------------------------
Significance level    : 0.05
Confidence level      : 0.95
Bootstrap iterations  : 10000
Master seed           : 2025
Statistical seed      : 3925
---------------------------------------------------------------------------
Validation checks     : 9
Validation passed     : 9
Validation failed     : 0
Section 1 status      : PASS
--------------------------------

In [22]:
# ============================================================
# SECTION 2 — LOAD FINAL EVALUATION RESULTS
# ============================================================

# ------------------------------------------------------------
# 1. LOAD AUTHORITATIVE NOTEBOOK 18 SOURCE
# ------------------------------------------------------------

NOTEBOOK_18_ROOT = (
    PROJECT_ROOT
    / "results"
    / "notebooks"
    / "notebook_18"
)

SOURCE_PATH = (
    NOTEBOOK_18_ROOT
    / "results"
    / "tradeoff_measures.csv"
)

if not SOURCE_PATH.is_file():
    raise FileNotFoundError(
        f"Notebook 18 source not found: {SOURCE_PATH}"
    )

evaluation_df = pd.read_csv(SOURCE_PATH)

if evaluation_df.empty:
    raise ValueError("Notebook 18 evaluation source is empty.")

# Preserve original column names for audit.
original_columns = evaluation_df.columns.tolist()

evaluation_df.columns = [
    str(column).strip().lower()
    for column in evaluation_df.columns
]

print("=" * 75)
print("NOTEBOOK 19 — SECTION 2: SOURCE SCHEMA INSPECTION")
print("=" * 75)
print("Source:", SOURCE_PATH)
print("Rows:", len(evaluation_df))
print("Original columns:")
for column in original_columns:
    print(" -", column)

print("-" * 75)
print("Normalized columns:")
for column in evaluation_df.columns:
    print(" -", column)

print("-" * 75)
print("First five rows:")
display(evaluation_df.head())

# ------------------------------------------------------------
# 2. NORMALIZE EXPLICIT IDENTIFIER ALIASES
# ------------------------------------------------------------

COLUMN_ALIASES = {
    "dataset": [
        "dataset",
        "dataset_name",
        "data_name",
    ],
    "variant": [
        "variant",
        "model_variant",
        "model",
        "method",
        "strategy",
    ],
    "repetition": [
        "repetition",
        "repetition_id",
        "repeat",
        "rep",
    ],
}

rename_map = {}

for standard_name, aliases in COLUMN_ALIASES.items():

    matches = [
        alias
        for alias in aliases
        if alias in evaluation_df.columns
    ]

    if len(matches) == 1:
        rename_map[matches[0]] = standard_name

    elif len(matches) > 1:
        raise ValueError(
            f"Ambiguous columns for {standard_name}: {matches}"
        )

evaluation_df = evaluation_df.rename(
    columns=rename_map
)

# ------------------------------------------------------------
# 3. VALIDATE EXPERIMENT IDENTIFIERS
# ------------------------------------------------------------

required_keys = [
    "dataset",
    "variant",
    "repetition",
]

missing_key_columns = [
    column
    for column in required_keys
    if column not in evaluation_df.columns
]

print("-" * 75)
print("Identifier mapping:", rename_map)
print("Missing required identifiers:", missing_key_columns)

if missing_key_columns:

    # Save inspection evidence before stopping.
    schema_report = {
        "source_path": str(SOURCE_PATH),
        "loaded_rows": int(len(evaluation_df)),
        "original_columns": original_columns,
        "normalized_columns": evaluation_df.columns.tolist(),
        "identifier_mapping": rename_map,
        "missing_required_identifiers": missing_key_columns,
        "status": "BLOCKED_SCHEMA_MISMATCH",
    }

    schema_report_path = (
        METADATA_DIR
        / "section_02_schema_diagnostic.json"
    )

    with open(
        schema_report_path,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            schema_report,
            file,
            indent=4
        )

    print(
        "Schema diagnostic saved:",
        schema_report_path
    )

    raise ValueError(
        "Section 2 is blocked. The actual source columns do not "
        "provide all three required experiment identifiers. "
        "Inspect the printed columns and diagnostic before proceeding."
    )

# ------------------------------------------------------------
# 4. NORMALIZE IDENTIFIER VALUES
# ------------------------------------------------------------

evaluation_df["dataset"] = (
    evaluation_df["dataset"]
    .astype("string")
    .str.strip()
)

evaluation_df["variant"] = (
    evaluation_df["variant"]
    .astype("string")
    .str.strip()
)

evaluation_df["repetition"] = pd.to_numeric(
    evaluation_df["repetition"],
    errors="coerce"
)

invalid_key_rows = evaluation_df[
    evaluation_df[required_keys].isna().any(axis=1)
].copy()

valid_key_df = evaluation_df.dropna(
    subset=required_keys
).copy()

valid_key_df["repetition"] = (
    valid_key_df["repetition"].astype(int)
)

# ------------------------------------------------------------
# 5. VALIDATE EXPECTED EXPERIMENT DESIGN
# ------------------------------------------------------------

valid_key_df = valid_key_df[
    valid_key_df["dataset"].isin(DATASETS)
    & valid_key_df["variant"].isin(VARIANTS)
    & valid_key_df["repetition"].isin(REPETITIONS)
].copy()

duplicate_key_rows = valid_key_df[
    valid_key_df.duplicated(
        subset=required_keys,
        keep=False
    )
].copy()

expected_keys = {
    (dataset, variant, repetition)
    for dataset in DATASETS
    for variant in VARIANTS
    for repetition in REPETITIONS
}

observed_keys = set(
    map(
        tuple,
        valid_key_df[required_keys].itertuples(
            index=False,
            name=None
        )
    )
)

missing_expected_keys = sorted(
    expected_keys - observed_keys
)

unexpected_keys = sorted(
    observed_keys - expected_keys
)

# ------------------------------------------------------------
# 6. IDENTIFY AVAILABLE METRICS
# ------------------------------------------------------------

candidate_metrics = [
    "fidelity_score",
    "utility_score",
    "privacy_score",
    "achieved_epsilon",
    "achieved_delta",
    "wasserstein_distance",
    "js_divergence",
    "correlation_difference",
    "association_difference",
    "accuracy",
    "macro_f1",
    "weighted_f1",
    "membership_inference_risk",
    "reidentification_risk",
    "disclosure_risk",
    "nearest_neighbor_disclosure_risk",
]

available_metric_columns = [
    column
    for column in candidate_metrics
    if column in evaluation_df.columns
]

metric_coverage = {}

for column in available_metric_columns:

    numeric_values = pd.to_numeric(
        evaluation_df[column],
        errors="coerce"
    )

    metric_coverage[column] = {
        "non_missing": int(numeric_values.notna().sum()),
        "missing": int(numeric_values.isna().sum()),
        "unique_values": int(numeric_values.nunique()),
    }

# ------------------------------------------------------------
# 7. SAVE INPUT SNAPSHOT
# ------------------------------------------------------------

evaluation_snapshot_path = (
    RESULTS_DIR
    / "notebook_18_evaluation_snapshot.csv"
)

evaluation_df.to_csv(
    evaluation_snapshot_path,
    index=False
)

# ------------------------------------------------------------
# 8. SAVE VALIDATION AND SOURCE METADATA
# ------------------------------------------------------------

section_02_checks = [
    {
        "check": "source_loaded",
        "passed": True,
        "detail": str(SOURCE_PATH),
    },
    {
        "check": "expected_row_count",
        "passed": len(evaluation_df) == EXPECTED_RECORDS,
        "detail": f"Observed={len(evaluation_df)}",
    },
    {
        "check": "required_identifiers",
        "passed": not missing_key_columns,
        "detail": str(required_keys),
    },
    {
        "check": "expected_key_coverage",
        "passed": (
            len(observed_keys) == EXPECTED_RECORDS
            and not missing_expected_keys
            and not unexpected_keys
        ),
        "detail": (
            f"Observed={len(observed_keys)}; "
            f"Expected={EXPECTED_RECORDS}"
        ),
    },
    {
        "check": "no_duplicate_keys",
        "passed": duplicate_key_rows.empty,
        "detail": f"Duplicate rows={len(duplicate_key_rows)}",
    },
    {
        "check": "snapshot_saved",
        "passed": evaluation_snapshot_path.is_file(),
        "detail": str(evaluation_snapshot_path),
    },
]

section_02_validation_df = pd.DataFrame(
    section_02_checks
)

section_02_validation_path = (
    VALIDATION_DIR
    / "section_02_validation.csv"
)

section_02_validation_df.to_csv(
    section_02_validation_path,
    index=False
)

section_02_summary = {
    "section": "02",
    "status": (
        "PASS"
        if section_02_validation_df["passed"].all()
        else "FAIL"
    ),
    "source_path": str(SOURCE_PATH),
    "loaded_rows": int(len(evaluation_df)),
    "loaded_columns": int(len(evaluation_df.columns)),
    "identifier_mapping": rename_map,
    "observed_keys": int(len(observed_keys)),
    "expected_keys": int(EXPECTED_RECORDS),
    "missing_keys": int(len(missing_expected_keys)),
    "duplicate_key_rows": int(len(duplicate_key_rows)),
    "invalid_key_rows": int(len(invalid_key_rows)),
    "available_metrics": available_metric_columns,
    "metric_coverage": metric_coverage,
    "snapshot_path": str(evaluation_snapshot_path),
    "validation_path": str(section_02_validation_path),
}

section_02_summary_path = (
    METADATA_DIR
    / "section_02_summary.json"
)

with open(
    section_02_summary_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        section_02_summary,
        file,
        indent=4,
        default=str
    )

# ------------------------------------------------------------
# 9. DISPLAY RESULTS
# ------------------------------------------------------------

print("=" * 75)
print("NOTEBOOK 19 — SECTION 2: LOAD FINAL EVALUATION RESULTS")
print("=" * 75)
print("Source:", SOURCE_PATH)
print("Loaded rows:", len(evaluation_df))
print("Loaded columns:", len(evaluation_df.columns))
print("Identifier mapping:", rename_map)
print("Observed experiment keys:", len(observed_keys))
print("Expected experiment keys:", EXPECTED_RECORDS)
print("Duplicate key rows:", len(duplicate_key_rows))
print("Missing expected keys:", len(missing_expected_keys))
print("Invalid key rows:", len(invalid_key_rows))
print("Available metrics:", available_metric_columns)
print("-" * 75)
print(
    "Validation passed:",
    int(section_02_validation_df["passed"].sum())
)
print(
    "Validation failed:",
    int((~section_02_validation_df["passed"]).sum())
)
print("Section 2 status:", section_02_summary["status"])
print("Snapshot:", evaluation_snapshot_path)
print("Metadata:", source_metadata_path if 'source_metadata_path' in globals() else section_02_summary_path)
print("Validation:", section_02_validation_path)
print("=" * 75)

assert section_02_validation_df["passed"].all(), (
    "Section 2 validation failed. Do not proceed to Section 3."
)

NOTEBOOK 19 — SECTION 2: SOURCE SCHEMA INSPECTION
Source: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_18/results/tradeoff_measures.csv
Rows: 60
Original columns:
 - dataset
 - variant
 - repetition_id
 - dp_configured
 - dp_applied
 - privacy_score
 - achieved_epsilon
 - formal_dp_verified
 - fidelity_score
 - utility_score
 - empirical_privacy_score
 - complete_three_way
 - tradeoff_score
 - privacy_fidelity_available
 - privacy_utility_available
 - fidelity_utility_available
---------------------------------------------------------------------------
Normalized columns:
 - dataset
 - variant
 - repetition_id
 - dp_configured
 - dp_applied
 - privacy_score
 - achieved_epsilon
 - formal_dp_verified
 - fidelity_score
 - utility_score
 - empirical_privacy_score
 - complete_three_way
 - tradeoff_score
 - privacy_fidelity_available
 - privacy_utility_available
 - fidelity_utility_available
---------------------------------------------------------------------------
Fir

,dataset,variant,repetition_id,dp_configured,dp_applied,privacy_score,achieved_epsilon,formal_dp_verified,fidelity_score,utility_score,empirical_privacy_score,complete_three_way,tradeoff_score,privacy_fidelity_available,privacy_utility_available,fidelity_utility_available
0,adult_income,gan_only,1,False,False,NaN,NaN,False,NaN,NaN,NaN,False,NaN,False,False,False
1,adult_income,gan_only,2,False,False,NaN,NaN,False,NaN,NaN,NaN,False,NaN,False,False,False
2,adult_income,gan_only,3,False,False,NaN,NaN,False,NaN,NaN,NaN,False,NaN,False,False,False
3,adult_income,gan_only,4,False,False,NaN,NaN,False,NaN,NaN,NaN,False,NaN,False,False,False
4,adult_income,gan_only,5,False,False,NaN,NaN,False,NaN,NaN,NaN,False,NaN,False,False,False


---------------------------------------------------------------------------
Identifier mapping: {'dataset': 'dataset', 'variant': 'variant', 'repetition_id': 'repetition'}
Missing required identifiers: []
NOTEBOOK 19 — SECTION 2: LOAD FINAL EVALUATION RESULTS
Source: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_18/results/tradeoff_measures.csv
Loaded rows: 60
Loaded columns: 16
Identifier mapping: {'dataset': 'dataset', 'variant': 'variant', 'repetition_id': 'repetition'}
Observed experiment keys: 60
Expected experiment keys: 60
Duplicate key rows: 0
Missing expected keys: 0
Invalid key rows: 0
Available metrics: ['fidelity_score', 'utility_score', 'privacy_score', 'achieved_epsilon']
---------------------------------------------------------------------------
Validation passed: 6
Validation failed: 0
Section 2 status: PASS
Snapshot: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/results/notebook_18_evaluation_snapshot.csv
Metadata: /content/

In [23]:
# ============================================================
# SECTION 3 — VALIDATE REPEATED EXPERIMENTS
# ============================================================

def normalize_column_names(df):

    df = df.copy()

    df.columns = [
        str(col).strip().lower()
        for col in df.columns
    ]

    return df


evaluation_df = normalize_column_names(
    evaluation_df
)

# ------------------------------------------------------------
# NORMALIZE KEY ALIASES
# ------------------------------------------------------------

KEY_ALIASES = {
    "dataset": [
        "dataset_name",
        "data_name"
    ],
    "variant": [
        "model",
        "model_variant",
        "method",
        "strategy"
    ],
    "repetition": [
        "repeat",
        "rep",
        "run",
        "run_id",
        "repetition_id"
    ]
}

for canonical, aliases in KEY_ALIASES.items():

    if canonical not in evaluation_df.columns:

        for alias in aliases:

            if alias in evaluation_df.columns:

                evaluation_df = evaluation_df.rename(
                    columns={alias: canonical}
                )

                break

# ------------------------------------------------------------
# REPEATED EXPERIMENT VALIDATION
# ------------------------------------------------------------

required_keys_present = all(
    col in evaluation_df.columns
    for col in KEY_COLUMNS
)

if required_keys_present:

    evaluation_df["dataset"] = (
        evaluation_df["dataset"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    evaluation_df["variant"] = (
        evaluation_df["variant"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    evaluation_df["repetition"] = pd.to_numeric(
        evaluation_df["repetition"],
        errors="coerce"
    )

    evaluation_df = evaluation_df.loc[
        evaluation_df["dataset"].isin(DATASETS)
        & evaluation_df["variant"].isin(VARIANTS)
        & evaluation_df["repetition"].isin(REPETITIONS)
    ].copy()

    evaluation_df["repetition"] = (
        evaluation_df["repetition"].astype("Int64")
    )

    duplicate_mask = evaluation_df.duplicated(
        KEY_COLUMNS,
        keep=False
    )

    duplicate_records = evaluation_df.loc[
        duplicate_mask
    ].copy()

    # Do not silently aggregate duplicate experimental records.
    valid_unique_df = evaluation_df.loc[
        ~duplicate_mask
    ].copy()

else:

    duplicate_records = pd.DataFrame()

    valid_unique_df = pd.DataFrame(
        columns=evaluation_df.columns
    )

# ------------------------------------------------------------
# EXPECTED EXPERIMENT GRID
# ------------------------------------------------------------

expected_grid_df = pd.DataFrame(

    itertools.product(
        DATASETS,
        VARIANTS,
        REPETITIONS
    ),

    columns=KEY_COLUMNS

)

if not valid_unique_df.empty:

    observed_keys = valid_unique_df[
        KEY_COLUMNS
    ].drop_duplicates()

    grid_check_df = expected_grid_df.merge(
        observed_keys,
        on=KEY_COLUMNS,
        how="left",
        indicator=True
    )

    missing_experiments_df = grid_check_df.loc[
        grid_check_df["_merge"] == "left_only",
        KEY_COLUMNS
    ].copy()

else:

    missing_experiments_df = expected_grid_df.copy()

missing_experiments_path = (
    TABLES_DIR / "missing_experimental_conditions.csv"
)

missing_experiments_df.to_csv(
    missing_experiments_path,
    index=False
)

print("=" * 75)
print("SECTION 3 — VALIDATE REPEATED EXPERIMENTS")
print("=" * 75)
print("Expected conditions :", EXPECTED_RECORDS)
print("Loaded conditions   :", len(evaluation_df))
print("Unique valid records:", len(valid_unique_df))
print("Duplicate records   :", len(duplicate_records))
print("Missing conditions  :", len(missing_experiments_df))
print("Key schema valid    :", required_keys_present)
print("=" * 75)

SECTION 3 — VALIDATE REPEATED EXPERIMENTS
Expected conditions : 60
Loaded conditions   : 60
Unique valid records: 60
Duplicate records   : 0
Missing conditions  : 0
Key schema valid    : True


In [27]:
# ============================================================
# SECTION 4 — DEFINE STATISTICAL TESTING PROTOCOL
# ============================================================

STATISTICAL_PROTOCOL = {

    "design": "Repeated matched experimental comparisons",

    "experimental_unit": "Independent training repetition",

    "blocking_variables": [
        "dataset",
        "repetition"
    ],

    "comparison_variable": "variant",

    "variants": VARIANTS,

    "primary_metrics": PRIMARY_METRICS,

    "empirical_privacy_metrics": (
        EMPIRICAL_PRIVACY_METRICS
    ),

    "formal_dp_metrics": [
        "achieved_epsilon",
        "achieved_delta"
    ],

    "paired_test": "Wilcoxon signed-rank test",

    "omnibus_test": "Friedman test",

    "post_hoc_test": (
        "Pairwise Wilcoxon signed-rank tests"
    ),

    "multiple_comparison_correction": "Holm",

    "effect_size": (
        "Paired rank-biserial correlation"
    ),

    "confidence_interval": (
        "Paired bootstrap percentile confidence interval"
    ),

    "significance_level": ALPHA,

    "confidence_level": CONFIDENCE_LEVEL,

    "repetitions_per_condition": len(REPETITIONS),

    "minimum_paired_repetitions": 3,

    "missing_data_policy": (
        "No imputation. Incomplete matched blocks are excluded "
        "from the relevant test and reported."
    ),

    "experimental_unit_policy": (
        "Repetitions are the independent experimental units. "
        "Rows, records, and features are not treated as independent "
        "experimental repetitions."
    ),

    "metric_direction_policy": {
        "higher_is_better": [
            "fidelity_score",
            "utility_score",
            "privacy_score",
            "accuracy",
            "macro_f1",
            "weighted_f1"
        ],
        "lower_is_better": [
            "wasserstein_distance",
            "js_divergence",
            "correlation_difference",
            "association_difference",
            "membership_inference_risk",
            "reidentification_risk",
            "disclosure_risk",
            "nearest_neighbor_disclosure_risk",
            "achieved_epsilon"
        ],
        "special_handling": (
            "Metric direction must be verified against the metric "
            "definition and source evaluation schema before interpreting "
            "effect signs. Missing or unverified metrics are not tested."
        )
    },

    "privacy_evidence_policy": (
        "Empirical privacy attack metrics and formally verified "
        "differential privacy accounting evidence are separate. "
        "Configured or applied DP flags do not establish a formal "
        "privacy guarantee."
    ),

    "five_repetition_limitation": (
        "With five nonzero paired observations, the minimum attainable "
        "two-sided exact Wilcoxon p-value is 0.0625. Therefore, "
        "conventional significance at alpha=0.05 cannot be established "
        "by an exact two-sided Wilcoxon test with only five pairs."
    ),

    "interpretation_policy": (
        "A non-significant result does not demonstrate equivalence. "
        "Report sample size, effect size, confidence interval, "
        "adjusted p-value, and test limitations."
    ),

    "confidence_interval_policy": (
        "Bootstrap intervals based on five repetitions are exploratory "
        "and must be interpreted cautiously."
    ),

    "minimum_data_policy": (
        "Tests require at least the configured minimum number of "
        "complete matched repetitions. Insufficient evidence is "
        "reported as unavailable rather than imputed."
    ),

    "statistical_seed": STATISTICAL_SEED,

    "bootstrap_iterations": BOOTSTRAP_ITERATIONS,

}

# ------------------------------------------------------------
# SAVE STATISTICAL PROTOCOL
# ------------------------------------------------------------

protocol_path = (
    METADATA_DIR
    / "statistical_testing_protocol.json"
)

with open(
    protocol_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        STATISTICAL_PROTOCOL,
        file,
        indent=4
    )

# ------------------------------------------------------------
# VALIDATE PROTOCOL
# ------------------------------------------------------------

protocol_checks = {
    "matched_experimental_design": (
        STATISTICAL_PROTOCOL["design"]
        == "Repeated matched experimental comparisons"
    ),
    "repetition_as_experimental_unit": (
        STATISTICAL_PROTOCOL["experimental_unit"]
        == "Independent training repetition"
    ),
    "wilcoxon_defined": (
        STATISTICAL_PROTOCOL["paired_test"]
        == "Wilcoxon signed-rank test"
    ),
    "friedman_defined": (
        STATISTICAL_PROTOCOL["omnibus_test"]
        == "Friedman test"
    ),
    "holm_correction_defined": (
        STATISTICAL_PROTOCOL["multiple_comparison_correction"]
        == "Holm"
    ),
    "privacy_evidence_separated": bool(
        STATISTICAL_PROTOCOL["privacy_evidence_policy"]
    ),
    "five_repetition_limitation_documented": bool(
        STATISTICAL_PROTOCOL["five_repetition_limitation"]
    ),
    "missing_data_policy_defined": bool(
        STATISTICAL_PROTOCOL["missing_data_policy"]
    ),
    "protocol_saved": protocol_path.is_file(),
}

protocol_validation_df = pd.DataFrame(
    [
        {
            "check": name,
            "passed": passed
        }
        for name, passed in protocol_checks.items()
    ]
)

protocol_validation_path = (
    VALIDATION_DIR
    / "section_04_protocol_validation.csv"
)

protocol_validation_df.to_csv(
    protocol_validation_path,
    index=False
)

print("=" * 75)
print("SECTION 4 — DEFINE STATISTICAL TESTING PROTOCOL")
print("=" * 75)
print("Validation passed:", sum(protocol_checks.values()))
print("Validation failed:", len(protocol_checks) - sum(protocol_checks.values()))
print("Significance level:", ALPHA)
print("Confidence level:", CONFIDENCE_LEVEL)
print("Independent repetitions:", len(REPETITIONS))
print("Paired test:", STATISTICAL_PROTOCOL["paired_test"])
print("Omnibus test:", STATISTICAL_PROTOCOL["omnibus_test"])
print("Multiple comparison correction:", "Holm")
print("Protocol saved:", protocol_path)
print("Validation saved:", protocol_validation_path)
print("=" * 75)

assert all(protocol_checks.values()), (
    "Section 4 protocol validation failed."
)

SECTION 4 — DEFINE STATISTICAL TESTING PROTOCOL
Validation passed: 9
Validation failed: 0
Significance level: 0.05
Confidence level: 0.95
Independent repetitions: 5
Paired test: Wilcoxon signed-rank test
Omnibus test: Friedman test
Multiple comparison correction: Holm
Protocol saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/metadata/statistical_testing_protocol.json
Validation saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/validation/section_04_protocol_validation.csv


In [29]:
# ============================================================
# SECTION 5 — PAIRWISE COMPARISONS
# ============================================================

PAIRWISE_RECORDS = []
pairwise_coverage_records = []

# ------------------------------------------------------------
# 1. DEFINE SAFE PAIRED METRIC EXTRACTION
# ------------------------------------------------------------

def get_metric_pairs(
    df,
    metric,
    dataset,
    variant_a,
    variant_b
):

    required_columns = [
        "dataset",
        "variant",
        "repetition",
        metric
    ]

    if not set(required_columns).issubset(df.columns):
        return pd.DataFrame()

    subset = df.loc[
        (df["dataset"] == dataset)
        & (df["variant"].isin([variant_a, variant_b]))
    ].copy()

    if subset.empty:
        return pd.DataFrame()

    subset[metric] = pd.to_numeric(
        subset[metric],
        errors="coerce"
    )

    subset = subset.replace(
        [np.inf, -np.inf],
        np.nan
    )

    # Never silently aggregate repeated experiment keys.
    duplicate_mask = subset.duplicated(
        subset=["dataset", "variant", "repetition"],
        keep=False
    )

    if duplicate_mask.any():
        raise ValueError(
            f"Duplicate experiment keys detected for "
            f"{dataset}, {metric}, "
            f"{variant_a} vs {variant_b}."
        )

    pivot = subset.pivot(
        index="repetition",
        columns="variant",
        values=metric
    )

    if (
        variant_a not in pivot.columns
        or variant_b not in pivot.columns
    ):
        return pd.DataFrame(
            columns=[variant_a, variant_b]
        )

    pairs = pivot[
        [variant_a, variant_b]
    ].dropna()

    return pairs

# ------------------------------------------------------------
# 2. IDENTIFY AVAILABLE METRICS
# ------------------------------------------------------------

available_metrics = [
    metric
    for metric in ALL_TEST_METRICS
    if metric in valid_unique_df.columns
]

# ------------------------------------------------------------
# 3. BUILD PAIRWISE EVIDENCE COVERAGE
# ------------------------------------------------------------

for dataset in DATASETS:

    for metric in available_metrics:

        for variant_a, variant_b in itertools.combinations(
            VARIANTS,
            2
        ):

            pairs = get_metric_pairs(
                valid_unique_df,
                metric,
                dataset,
                variant_a,
                variant_b
            )

            available_pair_count = len(pairs)

            if available_pair_count >= 3:
                status = "READY"
            elif available_pair_count > 0:
                status = "INSUFFICIENT_PAIRED_EVIDENCE"
            else:
                status = "NO_EMPIRICAL_METRIC_EVIDENCE"

            pairwise_coverage_records.append({

                "dataset": dataset,
                "metric": metric,
                "variant_a": variant_a,
                "variant_b": variant_b,

                "expected_pairs": len(REPETITIONS),

                "available_pairs": available_pair_count,

                "missing_pairs": (
                    len(REPETITIONS)
                    - available_pair_count
                ),

                "status": status,

                "test_executed": False,

                "interpretation": (
                    "Coverage assessment only; no statistical "
                    "test has been performed in this section."
                )

            })

# ------------------------------------------------------------
# 4. SAVE PAIRWISE COVERAGE
# ------------------------------------------------------------

pairwise_coverage_df = pd.DataFrame(
    pairwise_coverage_records
)

pairwise_coverage_path = (
    TABLES_DIR
    / "pairwise_comparison_coverage.csv"
)

pairwise_coverage_df.to_csv(
    pairwise_coverage_path,
    index=False
)

# ------------------------------------------------------------
# 5. COVERAGE SUMMARY
# ------------------------------------------------------------

if not pairwise_coverage_df.empty:

    ready_count = int(
        (
            pairwise_coverage_df["status"] == "READY"
        ).sum()
    )

    insufficient_count = int(
        (
            pairwise_coverage_df["status"]
            == "INSUFFICIENT_PAIRED_EVIDENCE"
        ).sum()
    )

    no_evidence_count = int(
        (
            pairwise_coverage_df["status"]
            == "NO_EMPIRICAL_METRIC_EVIDENCE"
        ).sum()
    )

else:

    ready_count = 0
    insufficient_count = 0
    no_evidence_count = 0

# ------------------------------------------------------------
# 6. VALIDATE SECTION 5
# ------------------------------------------------------------

section_05_checks = {
    "coverage_table_created": (
        not pairwise_coverage_df.empty
    ),

    "expected_comparison_count": (
        len(pairwise_coverage_df)
        == (
            len(DATASETS)
            * len(available_metrics)
            * len(list(itertools.combinations(VARIANTS, 2)))
        )
    ),

    "no_tests_executed_prematurely": (
        not pairwise_coverage_df["test_executed"].any()
        if not pairwise_coverage_df.empty
        else False
    ),

    "pair_counts_within_expected_range": (
        pairwise_coverage_df["available_pairs"].between(
            0,
            len(REPETITIONS)
        ).all()
        if not pairwise_coverage_df.empty
        else False
    ),

    "coverage_file_saved": (
        pairwise_coverage_path.is_file()
    ),
}

section_05_validation_df = pd.DataFrame([
    {
        "check": name,
        "passed": passed
    }
    for name, passed in section_05_checks.items()
])

section_05_validation_path = (
    VALIDATION_DIR
    / "section_05_validation.csv"
)

section_05_validation_df.to_csv(
    section_05_validation_path,
    index=False
)

# ------------------------------------------------------------
# 7. SAVE SECTION SUMMARY
# ------------------------------------------------------------

section_05_summary = {
    "section": "05",
    "status": (
        "PASS"
        if all(section_05_checks.values())
        else "FAIL"
    ),
    "available_metrics": available_metrics,
    "comparison_conditions": len(pairwise_coverage_df),
    "ready_comparisons": ready_count,
    "insufficient_comparisons": insufficient_count,
    "no_evidence_comparisons": no_evidence_count,
    "statistical_tests_executed": 0,
    "coverage_path": str(pairwise_coverage_path),
    "validation_path": str(section_05_validation_path),
}

section_05_summary_path = (
    METADATA_DIR
    / "section_05_summary.json"
)

with open(
    section_05_summary_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        section_05_summary,
        file,
        indent=4
    )

# ------------------------------------------------------------
# 8. DISPLAY RESULTS
# ------------------------------------------------------------

print("=" * 75)
print("NOTEBOOK 19 — SECTION 5: PAIRWISE COMPARISONS")
print("=" * 75)

print("Available metrics          :", available_metrics)
print("Pairwise comparison conditions:", len(pairwise_coverage_df))
print("Ready comparisons          :", ready_count)
print("Insufficient comparisons   :", insufficient_count)
print("No empirical metric evidence:", no_evidence_count)
print("Statistical tests executed :", 0)

print("-" * 75)

print(
    "Validation passed:",
    int(section_05_validation_df["passed"].sum())
)

print(
    "Validation failed:",
    int((~section_05_validation_df["passed"]).sum())
)

print("Section 5 status:", section_05_summary["status"])
print("Coverage saved:", pairwise_coverage_path)
print("Validation saved:", section_05_validation_path)
print("Summary saved:", section_05_summary_path)

print("=" * 75)

assert all(section_05_checks.values()), (
    "Section 5 validation failed."
)

NOTEBOOK 19 — SECTION 5: PAIRWISE COMPARISONS
Available metrics          : ['fidelity_score', 'utility_score', 'privacy_score']
Pairwise comparison conditions: 54
Ready comparisons          : 0
Insufficient comparisons   : 0
No empirical metric evidence: 54
Statistical tests executed : 0
---------------------------------------------------------------------------
Validation passed: 5
Validation failed: 0
Section 5 status: PASS
Coverage saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/tables/pairwise_comparison_coverage.csv
Validation saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/validation/section_05_validation.csv
Summary saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/metadata/section_05_summary.json


In [31]:
# ============================================================
# SECTION 6 — WILCOXON SIGNED-RANK TEST
# ============================================================

WILCOXON_RECORDS = []

# ------------------------------------------------------------
# 1. SAFE WILCOXON TEST FUNCTION
# ------------------------------------------------------------

def run_wilcoxon(x, y):

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if len(x) != len(y):
        raise ValueError(
            "Wilcoxon inputs must contain matched pairs."
        )

    valid = np.isfinite(x) & np.isfinite(y)

    x = x[valid]
    y = y[valid]

    differences = x - y
    n = len(differences)

    if n < STATISTICAL_PROTOCOL["minimum_paired_repetitions"]:

        return {
            "n_pairs": n,
            "statistic": np.nan,
            "p_value": np.nan,
            "test_method": "NOT_RUN",
            "status": (
                "NO_EMPIRICAL_METRIC_EVIDENCE"
                if n == 0
                else "INSUFFICIENT_PAIRS"
            ),
        }

    nonzero_differences = differences[
        ~np.isclose(differences, 0)
    ]

    if len(nonzero_differences) == 0:

        return {
            "n_pairs": n,
            "statistic": 0.0,
            "p_value": 1.0,
            "test_method": "DEGENERATE_ALL_ZERO",
            "status": "NO_DIFFERENCE",
        }

    # Exact two-sided inference is not conventionally significant
    # at alpha=0.05 with only five nonzero paired observations.
    # Explicitly use the asymptotic method only as an approximation.
    test_method = (
        "asymptotic"
        if len(nonzero_differences) <= 5
        else "auto"
    )

    try:

        result = stats.wilcoxon(
            x,
            y,
            alternative="two-sided",
            zero_method="wilcox",
            correction=False,
            method=test_method
        )

        return {
            "n_pairs": n,
            "statistic": float(result.statistic),
            "p_value": float(result.pvalue),
            "test_method": test_method,
            "status": "TESTED",
        }

    except Exception as exc:

        return {
            "n_pairs": n,
            "statistic": np.nan,
            "p_value": np.nan,
            "test_method": test_method,
            "status": "ERROR: " + str(exc)[:150],
        }

# ------------------------------------------------------------
# 2. EXECUTE PAIRWISE TESTS
# ------------------------------------------------------------

for dataset in DATASETS:

    for metric in available_metrics:

        for variant_a, variant_b in itertools.combinations(
            VARIANTS,
            2
        ):

            pairs = get_metric_pairs(
                valid_unique_df,
                metric,
                dataset,
                variant_a,
                variant_b
            )

            if pairs.empty:

                result = run_wilcoxon(
                    np.array([], dtype=float),
                    np.array([], dtype=float)
                )

            else:

                result = run_wilcoxon(
                    pairs[variant_a].values,
                    pairs[variant_b].values
                )

            WILCOXON_RECORDS.append({

                "dataset": dataset,
                "metric": metric,
                "variant_a": variant_a,
                "variant_b": variant_b,

                **result

            })

# ------------------------------------------------------------
# 3. CREATE RESULTS TABLE
# ------------------------------------------------------------

wilcoxon_df = pd.DataFrame(
    WILCOXON_RECORDS
)

wilcoxon_results_path = (
    TABLES_DIR
    / "wilcoxon_signed_rank_results.csv"
)

wilcoxon_df.to_csv(
    wilcoxon_results_path,
    index=False
)

# ------------------------------------------------------------
# 4. SUMMARIZE TEST STATUS
# ------------------------------------------------------------

status_counts = (
    wilcoxon_df["status"]
    .value_counts()
    .to_dict()
)

tested_count = int(
    (wilcoxon_df["status"] == "TESTED").sum()
)

unavailable_count = int(
    (
        wilcoxon_df["status"]
        == "NO_EMPIRICAL_METRIC_EVIDENCE"
    ).sum()
)

insufficient_count = int(
    (
        wilcoxon_df["status"]
        == "INSUFFICIENT_PAIRS"
    ).sum()
)

# ------------------------------------------------------------
# 5. VALIDATE SECTION 6
# ------------------------------------------------------------

section_06_checks = {

    "expected_test_records": (
        len(wilcoxon_df)
        == (
            len(DATASETS)
            * len(available_metrics)
            * len(list(itertools.combinations(VARIANTS, 2)))
        )
    ),

    "no_unavailable_p_values": (
        wilcoxon_df.loc[
            wilcoxon_df["status"] != "TESTED",
            "p_value"
        ].isna().all()
    ),

    "valid_test_p_values": (
        wilcoxon_df.loc[
            wilcoxon_df["status"] == "TESTED",
            "p_value"
        ].between(0, 1).all()
    ),

    "no_tests_claimed_without_pairs": (
        not (
            (wilcoxon_df["n_pairs"] == 0)
            & (wilcoxon_df["status"] == "TESTED")
        ).any()
    ),

    "results_saved": wilcoxon_results_path.is_file(),

}

section_06_validation_df = pd.DataFrame([
    {
        "check": name,
        "passed": passed
    }
    for name, passed in section_06_checks.items()
])

section_06_validation_path = (
    VALIDATION_DIR
    / "section_06_validation.csv"
)

section_06_validation_df.to_csv(
    section_06_validation_path,
    index=False
)

# ------------------------------------------------------------
# 6. SAVE SECTION SUMMARY
# ------------------------------------------------------------

section_06_summary = {

    "section": "06",

    "status": (
        "PASS"
        if all(section_06_checks.values())
        else "FAIL"
    ),

    "tests_attempted": int(len(wilcoxon_df)),

    "tests_executed": tested_count,

    "no_empirical_evidence": unavailable_count,

    "insufficient_pairs": insufficient_count,

    "status_counts": status_counts,

    "results_path": str(wilcoxon_results_path),

    "validation_path": str(section_06_validation_path),

    "interpretation": (
        "No inferential conclusions are available where empirical "
        "paired metric observations are absent."
    ),

}

section_06_summary_path = (
    METADATA_DIR
    / "section_06_summary.json"
)

with open(
    section_06_summary_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        section_06_summary,
        file,
        indent=4,
        default=str
    )

# ------------------------------------------------------------
# 7. DISPLAY RESULTS
# ------------------------------------------------------------

print("=" * 75)
print("NOTEBOOK 19 — SECTION 6: WILCOXON SIGNED-RANK TEST")
print("=" * 75)

print("Tests attempted          :", len(wilcoxon_df))
print("Tests executed           :", tested_count)
print("No empirical evidence    :", unavailable_count)
print("Insufficient pairs       :", insufficient_count)
print("Status distribution      :", status_counts)

print("-" * 75)

print(
    "Validation passed:",
    int(section_06_validation_df["passed"].sum())
)

print(
    "Validation failed:",
    int((~section_06_validation_df["passed"]).sum())
)

print("Section 6 status:", section_06_summary["status"])
print("Results saved:", wilcoxon_results_path)
print("Validation saved:", section_06_validation_path)
print("Summary saved:", section_06_summary_path)

print("=" * 75)

assert all(section_06_checks.values()), (
    "Section 6 validation failed."
)

NOTEBOOK 19 — SECTION 6: WILCOXON SIGNED-RANK TEST
Tests attempted          : 54
Tests executed           : 0
No empirical evidence    : 54
Insufficient pairs       : 0
Status distribution      : {'NO_EMPIRICAL_METRIC_EVIDENCE': 54}
---------------------------------------------------------------------------
Validation passed: 5
Validation failed: 0
Section 6 status: PASS
Results saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/tables/wilcoxon_signed_rank_results.csv
Validation saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/validation/section_06_validation.csv
Summary saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/metadata/section_06_summary.json


In [33]:
# ============================================================
# SECTION 7 — FRIEDMAN TEST
# ============================================================

FRIEDMAN_RECORDS = []

# ------------------------------------------------------------
# 1. EXECUTE FRIEDMAN TESTS
# ------------------------------------------------------------

for dataset in DATASETS:

    dataset_df = valid_unique_df.loc[
        valid_unique_df["dataset"] == dataset
    ].copy()

    for metric in available_metrics:

        if metric not in dataset_df.columns:
            continue

        dataset_df[metric] = pd.to_numeric(
            dataset_df[metric],
            errors="coerce"
        )

        dataset_df[metric] = dataset_df[metric].replace(
            [np.inf, -np.inf],
            np.nan
        )

        # Check duplicate blocks before reshaping.
        duplicate_blocks = dataset_df.duplicated(
            subset=["repetition", "variant"],
            keep=False
        )

        if duplicate_blocks.any():

            FRIEDMAN_RECORDS.append({
                "dataset": dataset,
                "metric": metric,
                "n_blocks": 0,
                "statistic": np.nan,
                "p_value": np.nan,
                "status": "DUPLICATE_BLOCKS",
            })

            continue

        # Use pivot rather than pivot_table to prevent
        # silent aggregation of repeated observations.
        pivot = dataset_df.pivot(
            index="repetition",
            columns="variant",
            values=metric
        )

        missing_variants = [
            variant
            for variant in VARIANTS
            if variant not in pivot.columns
        ]

        if missing_variants:

            FRIEDMAN_RECORDS.append({
                "dataset": dataset,
                "metric": metric,
                "n_blocks": 0,
                "statistic": np.nan,
                "p_value": np.nan,
                "status": "MISSING_VARIANT",
            })

            continue

        complete = pivot[VARIANTS].dropna()

        n_blocks = len(complete)

        if n_blocks == 0:

            status = "NO_EMPIRICAL_METRIC_EVIDENCE"
            statistic = np.nan
            p_value = np.nan

        elif n_blocks < 3:

            status = "INSUFFICIENT_BLOCKS"
            statistic = np.nan
            p_value = np.nan

        else:

            # Detect identical measurements across all variants.
            all_equal = all(
                np.allclose(
                    complete.iloc[:, 0].values,
                    complete[col].values
                )
                for col in complete.columns[1:]
            )

            if all_equal:

                status = "NO_DIFFERENCE"
                statistic = 0.0
                p_value = 1.0

            else:

                try:

                    result = stats.friedmanchisquare(
                        *[
                            complete[variant].values
                            for variant in VARIANTS
                        ]
                    )

                    statistic = float(result.statistic)
                    p_value = float(result.pvalue)
                    status = "TESTED"

                except Exception as exc:

                    statistic = np.nan
                    p_value = np.nan
                    status = "ERROR: " + str(exc)[:150]

        FRIEDMAN_RECORDS.append({

            "dataset": dataset,
            "metric": metric,
            "n_blocks": n_blocks,
            "statistic": statistic,
            "p_value": p_value,
            "status": status,

        })

# ------------------------------------------------------------
# 2. CREATE RESULTS TABLE
# ------------------------------------------------------------

friedman_df = pd.DataFrame(
    FRIEDMAN_RECORDS
)

friedman_results_path = (
    TABLES_DIR / "friedman_test_results.csv"
)

friedman_df.to_csv(
    friedman_results_path,
    index=False
)

# ------------------------------------------------------------
# 3. SUMMARIZE TEST STATUS
# ------------------------------------------------------------

friedman_status_counts = (
    friedman_df["status"]
    .value_counts()
    .to_dict()
)

friedman_tested_count = int(
    (friedman_df["status"] == "TESTED").sum()
)

friedman_unavailable_count = int(
    (
        friedman_df["status"]
        == "NO_EMPIRICAL_METRIC_EVIDENCE"
    ).sum()
)

friedman_insufficient_count = int(
    (
        friedman_df["status"]
        == "INSUFFICIENT_BLOCKS"
    ).sum()
)

# ------------------------------------------------------------
# 4. VALIDATE SECTION 7
# ------------------------------------------------------------

expected_friedman_records = (
    len(DATASETS) * len(available_metrics)
)

section_07_checks = {

    "expected_dataset_metric_records": (
        len(friedman_df) == expected_friedman_records
    ),

    "no_unavailable_p_values": (
        friedman_df.loc[
            friedman_df["status"] != "TESTED",
            "p_value"
        ].isna().all()
    ),

    "valid_test_p_values": (
        friedman_df.loc[
            friedman_df["status"] == "TESTED",
            "p_value"
        ].between(0, 1).all()
    ),

    "block_counts_within_expected_range": (
        friedman_df["n_blocks"].between(
            0,
            len(REPETITIONS)
        ).all()
    ),

    "results_saved": friedman_results_path.is_file(),

}

section_07_validation_df = pd.DataFrame([
    {
        "check": name,
        "passed": passed
    }
    for name, passed in section_07_checks.items()
])

section_07_validation_path = (
    VALIDATION_DIR / "section_07_validation.csv"
)

section_07_validation_df.to_csv(
    section_07_validation_path,
    index=False
)

# ------------------------------------------------------------
# 5. SAVE SECTION SUMMARY
# ------------------------------------------------------------

section_07_summary = {

    "section": "07",

    "status": (
        "PASS"
        if all(section_07_checks.values())
        else "FAIL"
    ),

    "tests_attempted": int(len(friedman_df)),

    "tests_executed": friedman_tested_count,

    "no_empirical_evidence": friedman_unavailable_count,

    "insufficient_blocks": friedman_insufficient_count,

    "status_counts": friedman_status_counts,

    "results_path": str(friedman_results_path),

    "validation_path": str(section_07_validation_path),

    "interpretation": (
        "No omnibus statistical conclusions are available "
        "where complete empirical measurements across all "
        "four variants are absent."
    ),

}

section_07_summary_path = (
    METADATA_DIR / "section_07_summary.json"
)

with open(
    section_07_summary_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        section_07_summary,
        file,
        indent=4,
        default=str
    )

# ------------------------------------------------------------
# 6. DISPLAY RESULTS
# ------------------------------------------------------------

print("=" * 75)
print("NOTEBOOK 19 — SECTION 7: FRIEDMAN TEST")
print("=" * 75)

print("Friedman comparisons attempted:", len(friedman_df))
print("Successfully tested:", friedman_tested_count)
print("No empirical evidence:", friedman_unavailable_count)
print("Insufficient blocks:", friedman_insufficient_count)
print("Status distribution:", friedman_status_counts)

print("-" * 75)

print(
    "Validation passed:",
    int(section_07_validation_df["passed"].sum())
)

print(
    "Validation failed:",
    int((~section_07_validation_df["passed"]).sum())
)

print("Section 7 status:", section_07_summary["status"])
print("Results saved:", friedman_results_path)
print("Validation saved:", section_07_validation_path)
print("Summary saved:", section_07_summary_path)

print("=" * 75)

assert all(section_07_checks.values()), (
    "Section 7 validation failed."
)

NOTEBOOK 19 — SECTION 7: FRIEDMAN TEST
Friedman comparisons attempted: 9
Successfully tested: 0
No empirical evidence: 9
Insufficient blocks: 0
Status distribution: {'NO_EMPIRICAL_METRIC_EVIDENCE': 9}
---------------------------------------------------------------------------
Validation passed: 5
Validation failed: 0
Section 7 status: PASS
Results saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/tables/friedman_test_results.csv
Validation saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/validation/section_07_validation.csv
Summary saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/metadata/section_07_summary.json


In [35]:
# ============================================================
# SECTION 8 — POST-HOC TESTS
# ============================================================

POSTHOC_RECORDS = []

# ------------------------------------------------------------
# 1. MATCH PAIRWISE RESULTS WITH OMNIBUS RESULTS
# ------------------------------------------------------------

for _, row in wilcoxon_df.iterrows():

    matching_friedman = friedman_df.loc[
        (friedman_df["dataset"] == row["dataset"])
        & (friedman_df["metric"] == row["metric"])
    ]

    if len(matching_friedman) > 1:
        raise ValueError(
            "Multiple Friedman records found for "
            f"{row['dataset']} / {row['metric']}."
        )

    if matching_friedman.empty:

        omnibus_p = np.nan
        omnibus_status = "OMNIBUS_RESULT_UNAVAILABLE"

    else:

        friedman_row = matching_friedman.iloc[0]

        omnibus_p = friedman_row["p_value"]
        omnibus_status = friedman_row["status"]

    omnibus_significant = (
        pd.notna(omnibus_p)
        and float(omnibus_p) < ALPHA
    )

    pairwise_test_executed = (
        row["status"] == "TESTED"
        and pd.notna(row["p_value"])
    )

    # --------------------------------------------------------
    # 2. CLASSIFY POST-HOC EVIDENCE
    # --------------------------------------------------------

    if not pairwise_test_executed:

        posthoc_status = "PAIRWISE_TEST_UNAVAILABLE"

        interpretation = (
            "No post-hoc inference: empirical paired "
            "measurements are unavailable."
        )

    elif omnibus_significant:

        posthoc_status = "POSTHOC_AFTER_SIGNIFICANT_OMNIBUS"

        interpretation = (
            "Pairwise comparison follows a significant "
            "Friedman omnibus result."
        )

    elif pd.notna(omnibus_p):

        posthoc_status = "EXPLORATORY_NON_SIGNIFICANT_OMNIBUS"

        interpretation = (
            "Exploratory pairwise comparison; the omnibus "
            "Friedman test was not significant."
        )

    else:

        posthoc_status = "EXPLORATORY_OMNIBUS_UNAVAILABLE"

        interpretation = (
            "Exploratory pairwise comparison; omnibus "
            "evidence is unavailable."
        )

    POSTHOC_RECORDS.append({

        **row.to_dict(),

        "friedman_p_value": omnibus_p,

        "friedman_status": omnibus_status,

        "friedman_significant": omnibus_significant,

        "pairwise_test_executed": pairwise_test_executed,

        "posthoc_status": posthoc_status,

        "interpretation": interpretation,

    })

# ------------------------------------------------------------
# 3. CREATE POST-HOC RESULTS TABLE
# ------------------------------------------------------------

posthoc_df = pd.DataFrame(
    POSTHOC_RECORDS
)

posthoc_results_path = (
    TABLES_DIR
    / "posthoc_pairwise_results.csv"
)

posthoc_df.to_csv(
    posthoc_results_path,
    index=False
)

# ------------------------------------------------------------
# 4. SUMMARIZE POST-HOC COVERAGE
# ------------------------------------------------------------

posthoc_status_counts = (
    posthoc_df["posthoc_status"]
    .value_counts()
    .to_dict()
)

posthoc_executed_count = int(
    posthoc_df["pairwise_test_executed"].sum()
)

posthoc_unavailable_count = int(
    (
        posthoc_df["posthoc_status"]
        == "PAIRWISE_TEST_UNAVAILABLE"
    ).sum()
)

posthoc_after_omnibus_count = int(
    (
        posthoc_df["posthoc_status"]
        == "POSTHOC_AFTER_SIGNIFICANT_OMNIBUS"
    ).sum()
)

# ------------------------------------------------------------
# 5. VALIDATE SECTION 8
# ------------------------------------------------------------

section_08_checks = {

    "expected_posthoc_records": (
        len(posthoc_df) == len(wilcoxon_df)
    ),

    "all_pairwise_records_matched": (
        posthoc_df[
            ["dataset", "metric", "variant_a", "variant_b"]
        ].equals(
            wilcoxon_df[
                ["dataset", "metric", "variant_a", "variant_b"]
            ].reset_index(drop=True)
        )
    ),

    "unavailable_tests_not_claimed_executed": (
        not (
            (
                posthoc_df["status"] != "TESTED"
            )
            & posthoc_df["pairwise_test_executed"]
        ).any()
    ),

    "unavailable_omnibus_not_claimed_significant": (
        not (
            posthoc_df["friedman_p_value"].isna()
            & posthoc_df["friedman_significant"]
        ).any()
    ),

    "results_saved": posthoc_results_path.is_file(),

}

section_08_validation_df = pd.DataFrame([
    {
        "check": name,
        "passed": passed
    }
    for name, passed in section_08_checks.items()
])

section_08_validation_path = (
    VALIDATION_DIR
    / "section_08_validation.csv"
)

section_08_validation_df.to_csv(
    section_08_validation_path,
    index=False
)

# ------------------------------------------------------------
# 6. SAVE SECTION SUMMARY
# ------------------------------------------------------------

section_08_summary = {

    "section": "08",

    "status": (
        "PASS"
        if all(section_08_checks.values())
        else "FAIL"
    ),

    "posthoc_comparisons": int(len(posthoc_df)),

    "pairwise_tests_executed": posthoc_executed_count,

    "pairwise_tests_unavailable": posthoc_unavailable_count,

    "posthoc_after_significant_omnibus": (
        posthoc_after_omnibus_count
    ),

    "status_counts": posthoc_status_counts,

    "results_path": str(posthoc_results_path),

    "validation_path": str(section_08_validation_path),

    "interpretation": (
        "Post-hoc records document pairwise evidence and "
        "omnibus-test availability. No inferential conclusions "
        "are drawn from unavailable empirical measurements."
    ),

}

section_08_summary_path = (
    METADATA_DIR
    / "section_08_summary.json"
)

with open(
    section_08_summary_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        section_08_summary,
        file,
        indent=4,
        default=str
    )

# ------------------------------------------------------------
# 7. DISPLAY RESULTS
# ------------------------------------------------------------

print("=" * 75)
print("NOTEBOOK 19 — SECTION 8: POST-HOC TESTS")
print("=" * 75)

print("Post-hoc comparisons:", len(posthoc_df))
print("Pairwise tests executed:", posthoc_executed_count)
print("Pairwise tests unavailable:", posthoc_unavailable_count)
print("After significant omnibus:", posthoc_after_omnibus_count)
print("Status distribution:", posthoc_status_counts)

print("-" * 75)

print(
    "Validation passed:",
    int(section_08_validation_df["passed"].sum())
)

print(
    "Validation failed:",
    int((~section_08_validation_df["passed"]).sum())
)

print("Section 8 status:", section_08_summary["status"])
print("Results saved:", posthoc_results_path)
print("Validation saved:", section_08_validation_path)
print("Summary saved:", section_08_summary_path)

print("=" * 75)

assert all(section_08_checks.values()), (
    "Section 8 validation failed."
)

NOTEBOOK 19 — SECTION 8: POST-HOC TESTS
Post-hoc comparisons: 54
Pairwise tests executed: 0
Pairwise tests unavailable: 54
After significant omnibus: 0
Status distribution: {'PAIRWISE_TEST_UNAVAILABLE': 54}
---------------------------------------------------------------------------
Validation passed: 5
Validation failed: 0
Section 8 status: PASS
Results saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/tables/posthoc_pairwise_results.csv
Validation saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/validation/section_08_validation.csv
Summary saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/metadata/section_08_summary.json


In [37]:
# ============================================================
# SECTION 9 — EFFECT SIZE
# ============================================================

EFFECT_SIZE_RECORDS = []

# ------------------------------------------------------------
# 1. PAIRED RANK-BISERIAL CORRELATION
# ------------------------------------------------------------

def paired_rank_biserial(x, y):

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if len(x) != len(y):
        raise ValueError(
            "Effect-size inputs must contain matched pairs."
        )

    valid = np.isfinite(x) & np.isfinite(y)

    differences = (x - y)[valid]

    differences = differences[
        ~np.isclose(differences, 0)
    ]

    if len(differences) == 0:
        return np.nan

    ranks = stats.rankdata(
        np.abs(differences),
        method="average"
    )

    positive_rank_sum = float(
        ranks[differences > 0].sum()
    )

    negative_rank_sum = float(
        ranks[differences < 0].sum()
    )

    denominator = (
        positive_rank_sum
        + negative_rank_sum
    )

    if denominator == 0:
        return np.nan

    return float(
        (positive_rank_sum - negative_rank_sum)
        / denominator
    )

# ------------------------------------------------------------
# 2. CALCULATE EFFECT SIZES
# ------------------------------------------------------------

for dataset in DATASETS:

    for metric in available_metrics:

        for variant_a, variant_b in itertools.combinations(
            VARIANTS,
            2
        ):

            pairs = get_metric_pairs(
                valid_unique_df,
                metric,
                dataset,
                variant_a,
                variant_b
            )

            n_pairs = len(pairs)

            if n_pairs >= STATISTICAL_PROTOCOL[
                "minimum_paired_repetitions"
            ]:

                values_a = pairs[variant_a].to_numpy(
                    dtype=float
                )

                values_b = pairs[variant_b].to_numpy(
                    dtype=float
                )

                differences = values_a - values_b

                mean_difference = float(
                    np.mean(differences)
                )

                median_difference = float(
                    np.median(differences)
                )

                effect = paired_rank_biserial(
                    values_a,
                    values_b
                )

                effect_status = (
                    "AVAILABLE"
                    if pd.notna(effect)
                    else "ALL_ZERO_DIFFERENCES"
                )

            else:

                mean_difference = np.nan
                median_difference = np.nan
                effect = np.nan

                effect_status = (
                    "NO_EMPIRICAL_METRIC_EVIDENCE"
                    if n_pairs == 0
                    else "INSUFFICIENT_PAIRED_EVIDENCE"
                )

            EFFECT_SIZE_RECORDS.append({

                "dataset": dataset,

                "metric": metric,

                "variant_a": variant_a,

                "variant_b": variant_b,

                "n_pairs": n_pairs,

                "mean_paired_difference": mean_difference,

                "median_paired_difference": median_difference,

                "rank_biserial_correlation": effect,

                "effect_direction": (
                    f"{variant_a} minus {variant_b}"
                ),

                "effect_status": effect_status,

            })

# ------------------------------------------------------------
# 3. CREATE EFFECT-SIZE TABLE
# ------------------------------------------------------------

effect_size_df = pd.DataFrame(
    EFFECT_SIZE_RECORDS
)

effect_size_path = (
    TABLES_DIR
    / "paired_effect_sizes.csv"
)

effect_size_df.to_csv(
    effect_size_path,
    index=False
)

# ------------------------------------------------------------
# 4. EFFECT-SIZE COVERAGE
# ------------------------------------------------------------

effect_available_count = int(
    (
        effect_size_df["effect_status"] == "AVAILABLE"
    ).sum()
)

effect_unavailable_count = int(
    (
        effect_size_df["effect_status"]
        == "NO_EMPIRICAL_METRIC_EVIDENCE"
    ).sum()
)

effect_zero_count = int(
    (
        effect_size_df["effect_status"]
        == "ALL_ZERO_DIFFERENCES"
    ).sum()
)

effect_insufficient_count = int(
    (
        effect_size_df["effect_status"]
        == "INSUFFICIENT_PAIRED_EVIDENCE"
    ).sum()
)

# ------------------------------------------------------------
# 5. VALIDATE SECTION 9
# ------------------------------------------------------------

expected_effect_records = (
    len(DATASETS)
    * len(available_metrics)
    * len(list(itertools.combinations(VARIANTS, 2)))
)

section_09_checks = {

    "expected_effect_size_records": (
        len(effect_size_df) == expected_effect_records
    ),

    "effect_sizes_within_valid_range": (
        effect_size_df[
            "rank_biserial_correlation"
        ].dropna().between(-1, 1).all()
    ),

    "unavailable_effects_not_fabricated": (
        effect_size_df.loc[
            effect_size_df["n_pairs"] == 0,
            [
                "mean_paired_difference",
                "median_paired_difference",
                "rank_biserial_correlation"
            ]
        ].isna().all().all()
    ),

    "pair_counts_within_expected_range": (
        effect_size_df["n_pairs"].between(
            0,
            len(REPETITIONS)
        ).all()
    ),

    "effect_size_file_saved": effect_size_path.is_file(),

}

section_09_validation_df = pd.DataFrame([
    {
        "check": name,
        "passed": passed
    }
    for name, passed in section_09_checks.items()
])

section_09_validation_path = (
    VALIDATION_DIR
    / "section_09_validation.csv"
)

section_09_validation_df.to_csv(
    section_09_validation_path,
    index=False
)

# ------------------------------------------------------------
# 6. SAVE SECTION SUMMARY
# ------------------------------------------------------------

section_09_summary = {

    "section": "09",

    "status": (
        "PASS"
        if all(section_09_checks.values())
        else "FAIL"
    ),

    "effect_size_comparisons": int(
        len(effect_size_df)
    ),

    "available_effect_sizes": effect_available_count,

    "unavailable_effect_sizes": effect_unavailable_count,

    "all_zero_difference_cases": effect_zero_count,

    "insufficient_pair_cases": effect_insufficient_count,

    "effect_size_definition": (
        "Paired rank-biserial correlation"
    ),

    "direction": (
        "Variant A minus Variant B; positive values indicate "
        "higher values for Variant A."
    ),

    "results_path": str(effect_size_path),

    "validation_path": str(section_09_validation_path),

}

section_09_summary_path = (
    METADATA_DIR
    / "section_09_summary.json"
)

with open(
    section_09_summary_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        section_09_summary,
        file,
        indent=4,
        default=str
    )

# ------------------------------------------------------------
# 7. DISPLAY RESULTS
# ------------------------------------------------------------

print("=" * 75)
print("NOTEBOOK 19 — SECTION 9: EFFECT SIZE")
print("=" * 75)

print("Effect-size comparisons:", len(effect_size_df))
print("Available effect sizes:", effect_available_count)
print("Unavailable effect sizes:", effect_unavailable_count)
print("All-zero difference cases:", effect_zero_count)
print("Insufficient pair cases:", effect_insufficient_count)

print("-" * 75)

print(
    "Validation passed:",
    int(section_09_validation_df["passed"].sum())
)

print(
    "Validation failed:",
    int((~section_09_validation_df["passed"]).sum())
)

print("Section 9 status:", section_09_summary["status"])
print("Results saved:", effect_size_path)
print("Validation saved:", section_09_validation_path)
print("Summary saved:", section_09_summary_path)

print("=" * 75)

assert all(section_09_checks.values()), (
    "Section 9 validation failed."
)

NOTEBOOK 19 — SECTION 9: EFFECT SIZE
Effect-size comparisons: 54
Available effect sizes: 0
Unavailable effect sizes: 54
All-zero difference cases: 0
Insufficient pair cases: 0
---------------------------------------------------------------------------
Validation passed: 5
Validation failed: 0
Section 9 status: PASS
Results saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/tables/paired_effect_sizes.csv
Validation saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/validation/section_09_validation.csv
Summary saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/metadata/section_09_summary.json


In [39]:
# ============================================================
# SECTION 10 — CONFIDENCE INTERVALS
# ============================================================

CI_RECORDS = []

# ------------------------------------------------------------
# 1. PAIRED BOOTSTRAP CONFIDENCE INTERVAL
# ------------------------------------------------------------

def paired_bootstrap_ci(
    x,
    y,
    iterations=BOOTSTRAP_ITERATIONS,
    confidence=CONFIDENCE_LEVEL,
    seed=STATISTICAL_SEED
):

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if len(x) != len(y):
        raise ValueError(
            "Confidence interval inputs must contain matched pairs."
        )

    valid = np.isfinite(x) & np.isfinite(y)

    differences = (x - y)[valid]

    n = len(differences)

    if n < STATISTICAL_PROTOCOL[
        "minimum_paired_repetitions"
    ]:
        return {
            "n_pairs": n,
            "mean_difference": np.nan,
            "ci_lower": np.nan,
            "ci_upper": np.nan,
            "status": (
                "NO_EMPIRICAL_METRIC_EVIDENCE"
                if n == 0
                else "INSUFFICIENT_PAIRED_EVIDENCE"
            )
        }

    rng = np.random.default_rng(seed)

    bootstrap_means = np.empty(
        iterations,
        dtype=float
    )

    for i in range(iterations):

        sample = rng.choice(
            differences,
            size=n,
            replace=True
        )

        bootstrap_means[i] = sample.mean()

    alpha_tail = (1 - confidence) / 2

    lower = float(
        np.quantile(
            bootstrap_means,
            alpha_tail
        )
    )

    upper = float(
        np.quantile(
            bootstrap_means,
            1 - alpha_tail
        )
    )

    observed_mean = float(
        differences.mean()
    )

    return {
        "n_pairs": n,
        "mean_difference": observed_mean,
        "ci_lower": lower,
        "ci_upper": upper,
        "status": "AVAILABLE"
    }

# ------------------------------------------------------------
# 2. CALCULATE CONFIDENCE INTERVALS
# ------------------------------------------------------------

for dataset in DATASETS:

    for metric in available_metrics:

        for variant_a, variant_b in itertools.combinations(
            VARIANTS,
            2
        ):

            pairs = get_metric_pairs(
                valid_unique_df,
                metric,
                dataset,
                variant_a,
                variant_b
            )

            seed = (
                STATISTICAL_SEED
                + DATASETS.index(dataset) * 100
                + ALL_TEST_METRICS.index(metric) * 10
                + VARIANTS.index(variant_a)
            )

            result = paired_bootstrap_ci(
                pairs[variant_a].values,
                pairs[variant_b].values,
                iterations=BOOTSTRAP_ITERATIONS,
                confidence=CONFIDENCE_LEVEL,
                seed=seed
            )

            CI_RECORDS.append({

                "dataset": dataset,
                "metric": metric,
                "variant_a": variant_a,
                "variant_b": variant_b,

                **result,

                "confidence_level": CONFIDENCE_LEVEL,
                "bootstrap_iterations": BOOTSTRAP_ITERATIONS,
                "random_seed": seed,

                "interpretation": (
                    "Exploratory interval; only five repetitions "
                    "are available per experimental condition."
                )

            })

# ------------------------------------------------------------
# 3. CREATE CONFIDENCE INTERVAL TABLE
# ------------------------------------------------------------

confidence_interval_df = pd.DataFrame(
    CI_RECORDS
)

confidence_interval_path = (
    TABLES_DIR
    / "paired_confidence_intervals.csv"
)

confidence_interval_df.to_csv(
    confidence_interval_path,
    index=False
)

# ------------------------------------------------------------
# 4. SUMMARIZE COVERAGE
# ------------------------------------------------------------

ci_available_count = int(
    (
        confidence_interval_df["status"] == "AVAILABLE"
    ).sum()
)

ci_unavailable_count = int(
    (
        confidence_interval_df["status"]
        == "NO_EMPIRICAL_METRIC_EVIDENCE"
    ).sum()
)

ci_insufficient_count = int(
    (
        confidence_interval_df["status"]
        == "INSUFFICIENT_PAIRED_EVIDENCE"
    ).sum()
)

# ------------------------------------------------------------
# 5. VALIDATE SECTION 10
# ------------------------------------------------------------

expected_ci_records = (
    len(DATASETS)
    * len(available_metrics)
    * len(list(itertools.combinations(VARIANTS, 2)))
)

available_ci_df = confidence_interval_df.loc[
    confidence_interval_df["status"] == "AVAILABLE"
]

section_10_checks = {

    "expected_ci_records": (
        len(confidence_interval_df) == expected_ci_records
    ),

    "valid_interval_bounds": (
        (
            available_ci_df["ci_lower"]
            <= available_ci_df["ci_upper"]
        ).all()
        if not available_ci_df.empty
        else True
    ),

    "unavailable_intervals_are_missing": (
        confidence_interval_df.loc[
            confidence_interval_df["status"]
            == "NO_EMPIRICAL_METRIC_EVIDENCE",
            [
                "mean_difference",
                "ci_lower",
                "ci_upper"
            ]
        ].isna().all().all()
    ),

    "pair_counts_within_expected_range": (
        confidence_interval_df["n_pairs"].between(
            0,
            len(REPETITIONS)
        ).all()
    ),

    "bootstrap_configuration_valid": (
        BOOTSTRAP_ITERATIONS > 0
        and 0 < CONFIDENCE_LEVEL < 1
    ),

    "results_saved": confidence_interval_path.is_file(),

}

section_10_validation_df = pd.DataFrame([
    {
        "check": name,
        "passed": passed
    }
    for name, passed in section_10_checks.items()
])

section_10_validation_path = (
    VALIDATION_DIR
    / "section_10_validation.csv"
)

section_10_validation_df.to_csv(
    section_10_validation_path,
    index=False
)

# ------------------------------------------------------------
# 6. SAVE SECTION SUMMARY
# ------------------------------------------------------------

section_10_summary = {

    "section": "10",

    "status": (
        "PASS"
        if all(section_10_checks.values())
        else "FAIL"
    ),

    "confidence_interval_comparisons": int(
        len(confidence_interval_df)
    ),

    "available_intervals": ci_available_count,

    "unavailable_intervals": ci_unavailable_count,

    "insufficient_pair_intervals": ci_insufficient_count,

    "confidence_level": CONFIDENCE_LEVEL,

    "bootstrap_iterations": BOOTSTRAP_ITERATIONS,

    "bootstrap_method": "Paired bootstrap percentile interval",

    "interpretation_caution": (
        "Bootstrap intervals based on five independent repetitions "
        "are exploratory and should not be interpreted as precise "
        "population-level uncertainty estimates."
    ),

    "results_path": str(confidence_interval_path),

    "validation_path": str(section_10_validation_path),

}

section_10_summary_path = (
    METADATA_DIR
    / "section_10_summary.json"
)

with open(
    section_10_summary_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        section_10_summary,
        file,
        indent=4,
        default=str
    )

# ------------------------------------------------------------
# 7. DISPLAY RESULTS
# ------------------------------------------------------------

print("=" * 75)
print("NOTEBOOK 19 — SECTION 10: CONFIDENCE INTERVALS")
print("=" * 75)

print("Confidence interval comparisons:", len(confidence_interval_df))
print("Available intervals:", ci_available_count)
print("No empirical evidence:", ci_unavailable_count)
print("Insufficient paired evidence:", ci_insufficient_count)
print("Confidence level:", CONFIDENCE_LEVEL)
print("Bootstrap iterations:", BOOTSTRAP_ITERATIONS)

print("-" * 75)

print(
    "Validation passed:",
    int(section_10_validation_df["passed"].sum())
)

print(
    "Validation failed:",
    int((~section_10_validation_df["passed"]).sum())
)

print("Section 10 status:", section_10_summary["status"])
print("Results saved:", confidence_interval_path)
print("Validation saved:", section_10_validation_path)
print("Summary saved:", section_10_summary_path)

print("=" * 75)

assert all(section_10_checks.values()), (
    "Section 10 validation failed."
)

NOTEBOOK 19 — SECTION 10: CONFIDENCE INTERVALS
Confidence interval comparisons: 54
Available intervals: 0
No empirical evidence: 54
Insufficient paired evidence: 0
Confidence level: 0.95
Bootstrap iterations: 10000
---------------------------------------------------------------------------
Validation passed: 6
Validation failed: 0
Section 10 status: PASS
Results saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/tables/paired_confidence_intervals.csv
Validation saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/validation/section_10_validation.csv
Summary saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/metadata/section_10_summary.json


In [41]:
# ============================================================
# SECTION 11 — MULTIPLE-COMPARISON CORRECTION
# ============================================================

# ------------------------------------------------------------
# 1. HOLM-BONFERRONI ADJUSTMENT
# ------------------------------------------------------------

def holm_adjust(p_values):

    p_values = np.asarray(
        p_values,
        dtype=float
    )

    adjusted = np.full(
        len(p_values),
        np.nan,
        dtype=float
    )

    valid_indices = np.where(
        np.isfinite(p_values)
    )[0]

    if len(valid_indices) == 0:
        return adjusted

    valid_p = p_values[valid_indices]

    if np.any((valid_p < 0) | (valid_p > 1)):
        raise ValueError(
            "P-values must be within [0, 1]."
        )

    order = np.argsort(valid_p)

    sorted_p = valid_p[order]

    m = len(sorted_p)

    adjusted_sorted = np.maximum.accumulate(
        [
            min(
                1.0,
                (m - i) * sorted_p[i]
            )
            for i in range(m)
        ]
    )

    adjusted_valid = np.empty(
        m,
        dtype=float
    )

    adjusted_valid[order] = adjusted_sorted

    adjusted[valid_indices] = adjusted_valid

    return adjusted

# ------------------------------------------------------------
# 2. POST-HOC PAIRWISE HOLM CORRECTION
# ------------------------------------------------------------

posthoc_df = posthoc_df.copy()

posthoc_df["p_adjusted_holm"] = np.nan
posthoc_df["significant_holm"] = False
posthoc_df["correction_family"] = (
    "dataset_metric_pairwise"
)

if not posthoc_df.empty:

    for (dataset, metric), indices in posthoc_df.groupby(
        ["dataset", "metric"]
    ).groups.items():

        group = posthoc_df.loc[indices]

        valid_test_mask = (
            group["status"].eq("TESTED")
            & group["p_value"].notna()
        )

        valid_indices = group.index[
            valid_test_mask
        ]

        if len(valid_indices) == 0:
            continue

        adjusted = holm_adjust(
            posthoc_df.loc[
                valid_indices,
                "p_value"
            ].values
        )

        posthoc_df.loc[
            valid_indices,
            "p_adjusted_holm"
        ] = adjusted

        posthoc_df.loc[
            valid_indices,
            "significant_holm"
        ] = adjusted < ALPHA

# ------------------------------------------------------------
# 3. FRIEDMAN OMNIBUS HOLM CORRECTION
# ------------------------------------------------------------

friedman_df = friedman_df.copy()

friedman_df["p_adjusted_holm"] = np.nan
friedman_df["significant_holm"] = False

friedman_df["correction_family"] = (
    "all_dataset_metric_omnibus_tests"
)

valid_friedman_mask = (
    friedman_df["status"].eq("TESTED")
    & friedman_df["p_value"].notna()
)

valid_friedman_indices = friedman_df.index[
    valid_friedman_mask
]

if len(valid_friedman_indices) > 0:

    adjusted = holm_adjust(
        friedman_df.loc[
            valid_friedman_indices,
            "p_value"
        ].values
    )

    friedman_df.loc[
        valid_friedman_indices,
        "p_adjusted_holm"
    ] = adjusted

    friedman_df.loc[
        valid_friedman_indices,
        "significant_holm"
    ] = adjusted < ALPHA

# ------------------------------------------------------------
# 4. SAVE CORRECTED RESULTS
# ------------------------------------------------------------

posthoc_results_path = (
    TABLES_DIR
    / "posthoc_pairwise_results.csv"
)

friedman_results_path = (
    TABLES_DIR
    / "friedman_test_results.csv"
)

posthoc_df.to_csv(
    posthoc_results_path,
    index=False
)

friedman_df.to_csv(
    friedman_results_path,
    index=False
)

# ------------------------------------------------------------
# 5. SUMMARIZE CORRECTION COVERAGE
# ------------------------------------------------------------

posthoc_adjusted_count = int(
    posthoc_df["p_adjusted_holm"].notna().sum()
)

friedman_adjusted_count = int(
    friedman_df["p_adjusted_holm"].notna().sum()
)

posthoc_significant_count = int(
    posthoc_df["significant_holm"].sum()
)

friedman_significant_count = int(
    friedman_df["significant_holm"].sum()
)

# ------------------------------------------------------------
# 6. VALIDATE SECTION 11
# ------------------------------------------------------------

section_11_checks = {

    "unavailable_posthoc_p_values_unadjusted": (
        posthoc_df.loc[
            posthoc_df["status"] != "TESTED",
            "p_adjusted_holm"
        ].isna().all()
    ),

    "unavailable_friedman_p_values_unadjusted": (
        friedman_df.loc[
            friedman_df["status"] != "TESTED",
            "p_adjusted_holm"
        ].isna().all()
    ),

    "adjusted_posthoc_p_values_valid": (
        posthoc_df["p_adjusted_holm"]
        .dropna()
        .between(0, 1)
        .all()
    ),

    "adjusted_friedman_p_values_valid": (
        friedman_df["p_adjusted_holm"]
        .dropna()
        .between(0, 1)
        .all()
    ),

    "significance_matches_adjusted_p_values": (
        (
            posthoc_df["significant_holm"]
            ==
            (
                posthoc_df["p_adjusted_holm"] < ALPHA
            ).fillna(False)
        ).all()
        and
        (
            friedman_df["significant_holm"]
            ==
            (
                friedman_df["p_adjusted_holm"] < ALPHA
            ).fillna(False)
        ).all()
    ),

    "corrected_results_saved": (
        posthoc_results_path.is_file()
        and friedman_results_path.is_file()
    ),

}

section_11_validation_df = pd.DataFrame([
    {
        "check": name,
        "passed": passed
    }
    for name, passed in section_11_checks.items()
])

section_11_validation_path = (
    VALIDATION_DIR
    / "section_11_validation.csv"
)

section_11_validation_df.to_csv(
    section_11_validation_path,
    index=False
)

# ------------------------------------------------------------
# 7. SAVE SECTION SUMMARY
# ------------------------------------------------------------

section_11_summary = {

    "section": "11",

    "status": (
        "PASS"
        if all(section_11_checks.values())
        else "FAIL"
    ),

    "correction_method": "Holm-Bonferroni",

    "alpha": ALPHA,

    "posthoc_comparisons": int(len(posthoc_df)),

    "posthoc_adjusted_tests": posthoc_adjusted_count,

    "posthoc_significant_tests": posthoc_significant_count,

    "friedman_comparisons": int(len(friedman_df)),

    "friedman_adjusted_tests": friedman_adjusted_count,

    "friedman_significant_tests": friedman_significant_count,

    "posthoc_correction_family": (
        "Separate family for each dataset and metric"
    ),

    "friedman_correction_family": (
        "All available dataset–metric omnibus tests"
    ),

    "interpretation": (
        "Unavailable tests retain missing adjusted p-values. "
        "No significance conclusions are generated without "
        "valid empirical test results."
    ),

    "posthoc_results_path": str(posthoc_results_path),

    "friedman_results_path": str(friedman_results_path),

    "validation_path": str(section_11_validation_path),

}

section_11_summary_path = (
    METADATA_DIR
    / "section_11_summary.json"
)

with open(
    section_11_summary_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        section_11_summary,
        file,
        indent=4,
        default=str
    )

# ------------------------------------------------------------
# 8. DISPLAY RESULTS
# ------------------------------------------------------------

print("=" * 75)
print("NOTEBOOK 19 — SECTION 11: MULTIPLE-COMPARISON CORRECTION")
print("=" * 75)

print("Correction method:", "Holm-Bonferroni")
print("Significance level:", ALPHA)

print("-" * 75)

print("Post-hoc comparisons:", len(posthoc_df))
print("Post-hoc adjusted tests:", posthoc_adjusted_count)
print("Post-hoc significant tests:", posthoc_significant_count)

print("-" * 75)

print("Friedman comparisons:", len(friedman_df))
print("Friedman adjusted tests:", friedman_adjusted_count)
print("Friedman significant tests:", friedman_significant_count)

print("-" * 75)

print(
    "Validation passed:",
    int(section_11_validation_df["passed"].sum())
)

print(
    "Validation failed:",
    int((~section_11_validation_df["passed"]).sum())
)

print("Section 11 status:", section_11_summary["status"])
print("Post-hoc results saved:", posthoc_results_path)
print("Friedman results saved:", friedman_results_path)
print("Validation saved:", section_11_validation_path)
print("Summary saved:", section_11_summary_path)

print("=" * 75)

assert all(section_11_checks.values()), (
    "Section 11 validation failed."
)

NOTEBOOK 19 — SECTION 11: MULTIPLE-COMPARISON CORRECTION
Correction method: Holm-Bonferroni
Significance level: 0.05
---------------------------------------------------------------------------
Post-hoc comparisons: 54
Post-hoc adjusted tests: 0
Post-hoc significant tests: 0
---------------------------------------------------------------------------
Friedman comparisons: 9
Friedman adjusted tests: 0
Friedman significant tests: 0
---------------------------------------------------------------------------
Validation passed: 6
Validation failed: 0
Section 11 status: PASS
Post-hoc results saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/tables/posthoc_pairwise_results.csv
Friedman results saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/tables/friedman_test_results.csv
Validation saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/validation/section_11_validation.csv
Summary saved: /content/drive/MyDrive/SPP_GA

In [43]:
# ============================================================
# SECTION 12 — SIGNIFICANCE MATRIX
# ============================================================

SIGNIFICANCE_MATRIX_RECORDS = []
SIGNIFICANCE_MATRIX_TABLES = {}

# ------------------------------------------------------------
# 1. GENERATE DATASET-METRIC SIGNIFICANCE MATRICES
# ------------------------------------------------------------

for dataset in DATASETS:

    for metric in available_metrics:

        matrix = pd.DataFrame(
            np.nan,
            index=VARIANTS,
            columns=VARIANTS,
            dtype=float
        )

        status_matrix = pd.DataFrame(
            "NOT_TESTED",
            index=VARIANTS,
            columns=VARIANTS,
            dtype=object
        )

        # Diagonal represents self-comparison.
        for variant in VARIANTS:

            matrix.loc[variant, variant] = 1.0
            status_matrix.loc[
                variant, variant
            ] = "SELF"

        subset = posthoc_df.loc[
            (posthoc_df["dataset"] == dataset)
            & (posthoc_df["metric"] == metric)
        ].copy()

        for _, row in subset.iterrows():

            a = row["variant_a"]
            b = row["variant_b"]

            if a not in VARIANTS or b not in VARIANTS:
                raise ValueError(
                    f"Unexpected variant in post-hoc results: {a}, {b}"
                )

            adjusted_p = row["p_adjusted_holm"]

            if (
                row["status"] == "TESTED"
                and pd.notna(adjusted_p)
            ):

                comparison_status = (
                    "SIGNIFICANT"
                    if float(adjusted_p) < ALPHA
                    else "NOT_SIGNIFICANT"
                )

                matrix.loc[a, b] = float(adjusted_p)
                matrix.loc[b, a] = float(adjusted_p)

                status_matrix.loc[a, b] = comparison_status
                status_matrix.loc[b, a] = comparison_status

            else:

                matrix.loc[a, b] = np.nan
                matrix.loc[b, a] = np.nan

                status_matrix.loc[a, b] = "NOT_TESTED"
                status_matrix.loc[b, a] = "NOT_TESTED"

        SIGNIFICANCE_MATRIX_TABLES[
            f"{dataset}_{metric}"
        ] = {
            "p_values": matrix,
            "status": status_matrix
        }

        # ----------------------------------------------------
        # 2. EXPORT LONG-FORM MATRIX RECORDS
        # ----------------------------------------------------

        for a in VARIANTS:

            for b in VARIANTS:

                p_value = matrix.loc[a, b]

                significance = status_matrix.loc[a, b]

                SIGNIFICANCE_MATRIX_RECORDS.append({

                    "dataset": dataset,

                    "metric": metric,

                    "variant_a": a,

                    "variant_b": b,

                    "adjusted_p_value": p_value,

                    "significance": significance,

                    "alpha": ALPHA,

                })

# ------------------------------------------------------------
# 3. CREATE AND SAVE MATRIX
# ------------------------------------------------------------

significance_matrix_df = pd.DataFrame(
    SIGNIFICANCE_MATRIX_RECORDS
)

significance_matrix_path = (
    TABLES_DIR / "significance_matrix.csv"
)

significance_matrix_df.to_csv(
    significance_matrix_path,
    index=False
)

# ------------------------------------------------------------
# 4. SAVE READABLE MATRIX TABLES
# ------------------------------------------------------------

matrix_export_records = []

for matrix_name, matrices in SIGNIFICANCE_MATRIX_TABLES.items():

    dataset, metric = matrix_name.rsplit("_", 1)

    for variant_a in VARIANTS:

        record = {
            "dataset": dataset,
            "metric": metric,
            "variant": variant_a,
        }

        for variant_b in VARIANTS:

            record[variant_b] = (
                matrices["status"].loc[
                    variant_a,
                    variant_b
                ]
            )

        matrix_export_records.append(record)

significance_matrix_wide_df = pd.DataFrame(
    matrix_export_records
)

significance_matrix_wide_path = (
    TABLES_DIR / "significance_matrix_wide.csv"
)

significance_matrix_wide_df.to_csv(
    significance_matrix_wide_path,
    index=False
)

# ------------------------------------------------------------
# 5. SUMMARIZE MATRIX COVERAGE
# ------------------------------------------------------------

significant_count = int(
    (
        significance_matrix_df["significance"]
        == "SIGNIFICANT"
    ).sum()
)

not_significant_count = int(
    (
        significance_matrix_df["significance"]
        == "NOT_SIGNIFICANT"
    ).sum()
)

not_tested_count = int(
    (
        significance_matrix_df["significance"]
        == "NOT_TESTED"
    ).sum()
)

self_count = int(
    (
        significance_matrix_df["significance"]
        == "SELF"
    ).sum()
)

# ------------------------------------------------------------
# 6. VALIDATE SECTION 12
# ------------------------------------------------------------

expected_entries = (
    len(DATASETS)
    * len(available_metrics)
    * len(VARIANTS)
    * len(VARIANTS)
)

symmetry_valid = True
diagonal_valid = True

for matrices in SIGNIFICANCE_MATRIX_TABLES.values():

    p_matrix = matrices["p_values"]
    s_matrix = matrices["status"]

    if not p_matrix.equals(p_matrix.T):
        symmetry_valid = False

    if not s_matrix.equals(s_matrix.T):
        symmetry_valid = False

    for variant in VARIANTS:

        if (
            p_matrix.loc[variant, variant] != 1.0
            or s_matrix.loc[variant, variant] != "SELF"
        ):
            diagonal_valid = False

section_12_checks = {

    "expected_matrix_entries": (
        len(significance_matrix_df) == expected_entries
    ),

    "matrix_symmetry": symmetry_valid,

    "diagonal_self_comparisons": diagonal_valid,

    "unavailable_tests_marked_not_tested": (
        significance_matrix_df.loc[
            significance_matrix_df["significance"] == "NOT_TESTED",
            "adjusted_p_value"
        ].isna().all()
    ),

    "valid_adjusted_p_values": (
        significance_matrix_df.loc[
            significance_matrix_df["adjusted_p_value"].notna(),
            "adjusted_p_value"
        ].between(0, 1).all()
    ),

    "both_matrix_files_saved": (
        significance_matrix_path.is_file()
        and significance_matrix_wide_path.is_file()
    ),

}

section_12_validation_df = pd.DataFrame([
    {
        "check": name,
        "passed": passed
    }
    for name, passed in section_12_checks.items()
])

section_12_validation_path = (
    VALIDATION_DIR / "section_12_validation.csv"
)

section_12_validation_df.to_csv(
    section_12_validation_path,
    index=False
)

# ------------------------------------------------------------
# 7. SAVE SECTION SUMMARY
# ------------------------------------------------------------

section_12_summary = {

    "section": "12",

    "status": (
        "PASS"
        if all(section_12_checks.values())
        else "FAIL"
    ),

    "matrix_groups": len(SIGNIFICANCE_MATRIX_TABLES),

    "matrix_entries": len(significance_matrix_df),

    "significant_entries": significant_count,

    "not_significant_entries": not_significant_count,

    "not_tested_entries": not_tested_count,

    "self_comparison_entries": self_count,

    "alpha": ALPHA,

    "interpretation": (
        "NOT_TESTED indicates unavailable inferential evidence. "
        "It must not be interpreted as a non-significant result."
    ),

    "long_matrix_path": str(significance_matrix_path),

    "wide_matrix_path": str(significance_matrix_wide_path),

    "validation_path": str(section_12_validation_path),

}

section_12_summary_path = (
    METADATA_DIR / "section_12_summary.json"
)

with open(
    section_12_summary_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        section_12_summary,
        file,
        indent=4,
        default=str
    )

# ------------------------------------------------------------
# 8. DISPLAY RESULTS
# ------------------------------------------------------------

print("=" * 75)
print("NOTEBOOK 19 — SECTION 12: SIGNIFICANCE MATRIX")
print("=" * 75)

print("Matrix groups:", len(SIGNIFICANCE_MATRIX_TABLES))
print("Matrix entries:", len(significance_matrix_df))
print("Significant entries:", significant_count)
print("Not significant entries:", not_significant_count)
print("Not tested entries:", not_tested_count)
print("Self comparisons:", self_count)

print("-" * 75)

print(
    "Validation passed:",
    int(section_12_validation_df["passed"].sum())
)

print(
    "Validation failed:",
    int((~section_12_validation_df["passed"]).sum())
)

print("Section 12 status:", section_12_summary["status"])
print("Long matrix saved:", significance_matrix_path)
print("Wide matrix saved:", significance_matrix_wide_path)
print("Validation saved:", section_12_validation_path)
print("Summary saved:", section_12_summary_path)

print("=" * 75)

assert all(section_12_checks.values()), (
    "Section 12 validation failed."
)

NOTEBOOK 19 — SECTION 12: SIGNIFICANCE MATRIX
Matrix groups: 9
Matrix entries: 144
Significant entries: 0
Not significant entries: 0
Not tested entries: 108
Self comparisons: 36
---------------------------------------------------------------------------
Validation passed: 6
Validation failed: 0
Section 12 status: PASS
Long matrix saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/tables/significance_matrix.csv
Wide matrix saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/tables/significance_matrix_wide.csv
Validation saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/validation/section_12_validation.csv
Summary saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/metadata/section_12_summary.json


In [45]:
# ============================================================
# SECTION 13 — DATASET-LEVEL SIGNIFICANCE
# ============================================================

DATASET_SIGNIFICANCE_RECORDS = []

# ------------------------------------------------------------
# 1. GENERATE DATASET-LEVEL SUMMARIES
# ------------------------------------------------------------

for dataset in DATASETS:

    for metric in available_metrics:

        friedman_subset = friedman_df.loc[
            (friedman_df["dataset"] == dataset)
            & (friedman_df["metric"] == metric)
        ].copy()

        # -----------------------------------------------
        # FRIEDMAN OMNIBUS RESULT
        # -----------------------------------------------

        if len(friedman_subset) == 0:

            omnibus_p = np.nan
            omnibus_adjusted = np.nan
            omnibus_status = "NO_EMPIRICAL_METRIC_EVIDENCE"

        elif len(friedman_subset) > 1:

            raise ValueError(
                f"Duplicate Friedman results for "
                f"{dataset} / {metric}"
            )

        else:

            omnibus_row = friedman_subset.iloc[0]

            omnibus_p = omnibus_row["p_value"]
            omnibus_adjusted = omnibus_row["p_adjusted_holm"]
            omnibus_status = omnibus_row["status"]

        # -----------------------------------------------
        # PAIRWISE POST-HOC RESULTS
        # -----------------------------------------------

        pair_subset = posthoc_df.loc[
            (posthoc_df["dataset"] == dataset)
            & (posthoc_df["metric"] == metric)
        ].copy()

        tested_pair_mask = (
            pair_subset["status"].eq("TESTED")
            & pair_subset["p_value"].notna()
        )

        valid_pair_tests = int(
            tested_pair_mask.sum()
        )

        significant_pair_tests = int(
            (
                tested_pair_mask
                & pair_subset["significant_holm"].eq(True)
            ).sum()
        )

        unavailable_pair_tests = int(
            (~tested_pair_mask).sum()
        )

        # -----------------------------------------------
        # EVIDENCE CLASSIFICATION
        # -----------------------------------------------

        if (
            omnibus_status == "TESTED"
            or valid_pair_tests > 0
        ):

            evidence_status = "TESTED"

        else:

            evidence_status = (
                "NO_EMPIRICAL_METRIC_EVIDENCE"
            )

        DATASET_SIGNIFICANCE_RECORDS.append({

            "dataset": dataset,

            "metric": metric,

            "friedman_p_value": omnibus_p,

            "friedman_adjusted_p": omnibus_adjusted,

            "friedman_status": omnibus_status,

            "valid_pairwise_tests": valid_pair_tests,

            "significant_pairwise_tests": (
                significant_pair_tests
            ),

            "unavailable_pairwise_tests": (
                unavailable_pair_tests
            ),

            "dataset_evidence_status": evidence_status,

        })

# ------------------------------------------------------------
# 2. CREATE DATASET-LEVEL RESULTS
# ------------------------------------------------------------

dataset_significance_df = pd.DataFrame(
    DATASET_SIGNIFICANCE_RECORDS
)

dataset_significance_path = (
    TABLES_DIR / "dataset_level_significance.csv"
)

dataset_significance_df.to_csv(
    dataset_significance_path,
    index=False
)

# ------------------------------------------------------------
# 3. VALIDATE RESULTS
# ------------------------------------------------------------

expected_summaries = (
    len(DATASETS) * len(available_metrics)
)

section_13_checks = {

    "expected_dataset_metric_summaries": (
        len(dataset_significance_df) == expected_summaries
    ),

    "unique_dataset_metric_pairs": (
        not dataset_significance_df.duplicated(
            ["dataset", "metric"]
        ).any()
    ),

    "all_expected_datasets_present": (
        set(dataset_significance_df["dataset"]) == set(DATASETS)
    ),

    "all_expected_metrics_present": (
        set(dataset_significance_df["metric"])
        == set(available_metrics)
    ),

    "pairwise_counts_nonnegative": (
        (
            dataset_significance_df[
                [
                    "valid_pairwise_tests",
                    "significant_pairwise_tests",
                    "unavailable_pairwise_tests",
                ]
            ] >= 0
        ).all().all()
    ),

    "significant_tests_do_not_exceed_valid_tests": (
        (
            dataset_significance_df[
                "significant_pairwise_tests"
            ]
            <= dataset_significance_df[
                "valid_pairwise_tests"
            ]
        ).all()
    ),

    "output_file_saved": (
        dataset_significance_path.is_file()
    ),

}

section_13_validation_df = pd.DataFrame([
    {
        "check": name,
        "passed": passed
    }
    for name, passed in section_13_checks.items()
])

section_13_validation_path = (
    VALIDATION_DIR / "section_13_validation.csv"
)

section_13_validation_df.to_csv(
    section_13_validation_path,
    index=False
)

# ------------------------------------------------------------
# 4. SAVE SECTION SUMMARY
# ------------------------------------------------------------

section_13_summary = {

    "section": "13",

    "status": (
        "PASS"
        if all(section_13_checks.values())
        else "FAIL"
    ),

    "dataset_metric_summaries": len(dataset_significance_df),

    "valid_pairwise_tests": int(
        dataset_significance_df[
            "valid_pairwise_tests"
        ].sum()
    ),

    "significant_pairwise_tests": int(
        dataset_significance_df[
            "significant_pairwise_tests"
        ].sum()
    ),

    "unavailable_pairwise_tests": int(
        dataset_significance_df[
            "unavailable_pairwise_tests"
        ].sum()
    ),

    "datasets": DATASETS,

    "metrics": available_metrics,

    "interpretation": (
        "Dataset-level summaries aggregate available omnibus "
        "and pairwise inferential evidence. Missing empirical "
        "measurements are classified as unavailable, not as "
        "non-significant findings."
    ),

    "output_path": str(dataset_significance_path),

    "validation_path": str(section_13_validation_path),

}

section_13_summary_path = (
    METADATA_DIR / "section_13_summary.json"
)

with open(
    section_13_summary_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        section_13_summary,
        file,
        indent=4,
        default=str
    )

# ------------------------------------------------------------
# 5. DISPLAY RESULTS
# ------------------------------------------------------------

print("=" * 75)
print("NOTEBOOK 19 — SECTION 13: DATASET-LEVEL SIGNIFICANCE")
print("=" * 75)

print(
    "Dataset-level summaries:",
    len(dataset_significance_df)
)

print(
    "Valid pairwise tests:",
    section_13_summary["valid_pairwise_tests"]
)

print(
    "Significant pairwise tests:",
    section_13_summary["significant_pairwise_tests"]
)

print(
    "Unavailable pairwise tests:",
    section_13_summary["unavailable_pairwise_tests"]
)

print("-" * 75)

print(
    "Validation passed:",
    int(section_13_validation_df["passed"].sum())
)

print(
    "Validation failed:",
    int((~section_13_validation_df["passed"]).sum())
)

print(
    "Section 13 status:",
    section_13_summary["status"]
)

print("Results saved:", dataset_significance_path)
print("Validation saved:", section_13_validation_path)
print("Summary saved:", section_13_summary_path)

print("=" * 75)

assert all(section_13_checks.values()), (
    "Section 13 validation failed."
)

NOTEBOOK 19 — SECTION 13: DATASET-LEVEL SIGNIFICANCE
Dataset-level summaries: 9
Valid pairwise tests: 0
Significant pairwise tests: 0
Unavailable pairwise tests: 54
---------------------------------------------------------------------------
Validation passed: 7
Validation failed: 0
Section 13 status: PASS
Results saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/tables/dataset_level_significance.csv
Validation saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/validation/section_13_validation.csv
Summary saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/metadata/section_13_summary.json


In [47]:
# ============================================================
# SECTION 14 — MODEL-LEVEL SIGNIFICANCE
# ============================================================

MODEL_LEVEL_METRICS = [
    metric
    for metric in PRIMARY_METRICS
    if metric in valid_unique_df.columns
]

MODEL_SIGNIFICANCE_RECORDS = []

model_data = valid_unique_df.copy()

# ------------------------------------------------------------
# 1. VALIDATE POOLED EXPERIMENTAL BLOCKS
# ------------------------------------------------------------

required_columns = [
    "dataset",
    "repetition",
    "variant"
]

missing_columns = [
    column
    for column in required_columns
    if column not in model_data.columns
]

if missing_columns:
    raise ValueError(
        f"Missing model-level key columns: {missing_columns}"
    )

model_data["block_id"] = (
    model_data["dataset"].astype(str)
    + "__rep_"
    + model_data["repetition"].astype(str)
)

duplicate_blocks = model_data.duplicated(
    ["dataset", "repetition", "variant"],
    keep=False
)

if duplicate_blocks.any():
    raise ValueError(
        "Duplicate dataset-repetition-variant records "
        "detected. Pooled testing cannot proceed."
    )

# ------------------------------------------------------------
# 2. EVALUATE EACH METRIC
# ------------------------------------------------------------

for metric in MODEL_LEVEL_METRICS:

    metric_data = model_data[
        [
            "dataset",
            "repetition",
            "block_id",
            "variant",
            metric
        ]
    ].copy()

    metric_data[metric] = pd.to_numeric(
        metric_data[metric],
        errors="coerce"
    )

    metric_data[metric] = metric_data[metric].replace(
        [np.inf, -np.inf],
        np.nan
    )

    observed_values = metric_data[metric].dropna()

    # --------------------------------------------------------
    # NO EMPIRICAL MEASUREMENTS
    # --------------------------------------------------------

    if observed_values.empty:

        MODEL_SIGNIFICANCE_RECORDS.append({

            "metric": metric,

            "test": "FRIEDMAN",

            "n_blocks": 0,

            "n_datasets": 0,

            "statistic": np.nan,

            "p_value": np.nan,

            "p_adjusted_holm": np.nan,

            "significant_holm": False,

            "status": "NO_EMPIRICAL_METRIC_EVIDENCE",

            "interpretation": (
                "No empirical measurements available."
            )

        })

        continue

    # --------------------------------------------------------
    # CHECK CROSS-DATASET COMPARABILITY
    # --------------------------------------------------------

    # Raw measurements from different datasets must not be
    # pooled unless metric definitions and scales are
    # demonstrably comparable.

    MODEL_SIGNIFICANCE_RECORDS.append({

        "metric": metric,

        "test": "FRIEDMAN",

        "n_blocks": 0,

        "n_datasets": int(
            metric_data["dataset"].nunique()
        ),

        "statistic": np.nan,

        "p_value": np.nan,

        "p_adjusted_holm": np.nan,

        "significant_holm": False,

        "status": "CROSS_DATASET_COMPARABILITY_NOT_ESTABLISHED",

        "interpretation": (
            "Pooled model-level significance testing was "
            "not performed because cross-dataset metric "
            "comparability has not been established."
        )

    })

# ------------------------------------------------------------
# 3. CREATE MODEL-LEVEL RESULTS
# ------------------------------------------------------------

model_significance_df = pd.DataFrame(
    MODEL_SIGNIFICANCE_RECORDS
)

# ------------------------------------------------------------
# 4. APPLY HOLM CORRECTION TO EXECUTED TESTS ONLY
# ------------------------------------------------------------

if not model_significance_df.empty:

    tested_mask = (
        model_significance_df["status"].eq("TESTED")
        & model_significance_df["p_value"].notna()
    )

    model_significance_df["p_adjusted_holm"] = np.nan

    model_significance_df.loc[
        tested_mask,
        "p_adjusted_holm"
    ] = holm_adjust(
        model_significance_df.loc[
            tested_mask,
            "p_value"
        ].values
    )

    model_significance_df["significant_holm"] = False

    model_significance_df.loc[
        tested_mask,
        "significant_holm"
    ] = (
        model_significance_df.loc[
            tested_mask,
            "p_adjusted_holm"
        ] < ALPHA
    )

# ------------------------------------------------------------
# 5. SAVE RESULTS
# ------------------------------------------------------------

model_significance_path = (
    TABLES_DIR / "model_level_significance.csv"
)

model_significance_df.to_csv(
    model_significance_path,
    index=False
)

# ------------------------------------------------------------
# 6. VALIDATE SECTION 14
# ------------------------------------------------------------

section_14_checks = {

    "expected_metric_records": (
        len(model_significance_df)
        == len(MODEL_LEVEL_METRICS)
    ),

    "unique_metric_records": (
        not model_significance_df["metric"].duplicated().any()
    ),

    "all_expected_metrics_present": (
        set(model_significance_df["metric"])
        == set(MODEL_LEVEL_METRICS)
    ),

    "no_unavailable_test_has_p_value": (
        model_significance_df.loc[
            model_significance_df["status"] != "TESTED",
            "p_value"
        ].isna().all()
    ),

    "adjusted_p_values_valid": (
        model_significance_df.loc[
            model_significance_df["p_adjusted_holm"].notna(),
            "p_adjusted_holm"
        ].between(0, 1).all()
    ),

    "output_file_saved": (
        model_significance_path.is_file()
    ),

}

section_14_validation_df = pd.DataFrame([
    {
        "check": name,
        "passed": passed
    }
    for name, passed in section_14_checks.items()
])

section_14_validation_path = (
    VALIDATION_DIR / "section_14_validation.csv"
)

section_14_validation_df.to_csv(
    section_14_validation_path,
    index=False
)

# ------------------------------------------------------------
# 7. SAVE SECTION SUMMARY
# ------------------------------------------------------------

section_14_summary = {

    "section": "14",

    "status": (
        "PASS"
        if all(section_14_checks.values())
        else "FAIL"
    ),

    "model_level_metrics": MODEL_LEVEL_METRICS,

    "metric_records": len(model_significance_df),

    "tests_executed": int(
        model_significance_df["status"].eq("TESTED").sum()
    ),

    "tests_unavailable": int(
        model_significance_df["status"].ne("TESTED").sum()
    ),

    "significant_tests": int(
        model_significance_df["significant_holm"].sum()
    ),

    "interpretation": (
        "Pooled model-level inferential testing was withheld "
        "because empirical measurements are unavailable or "
        "cross-dataset metric comparability is unestablished. "
        "No significance or ranking conclusion is inferred."
    ),

    "output_path": str(model_significance_path),

    "validation_path": str(section_14_validation_path),

}

section_14_summary_path = (
    METADATA_DIR / "section_14_summary.json"
)

with open(
    section_14_summary_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        section_14_summary,
        file,
        indent=4,
        default=str
    )

# ------------------------------------------------------------
# 8. DISPLAY RESULTS
# ------------------------------------------------------------

print("=" * 75)
print("NOTEBOOK 19 — SECTION 14: MODEL-LEVEL SIGNIFICANCE")
print("=" * 75)

print(
    "Model-level metric records:",
    len(model_significance_df)
)

print(
    "Tests executed:",
    section_14_summary["tests_executed"]
)

print(
    "Tests unavailable:",
    section_14_summary["tests_unavailable"]
)

print(
    "Significant tests:",
    section_14_summary["significant_tests"]
)

print("-" * 75)

print(
    "Validation passed:",
    int(section_14_validation_df["passed"].sum())
)

print(
    "Validation failed:",
    int((~section_14_validation_df["passed"]).sum())
)

print(
    "Section 14 status:",
    section_14_summary["status"]
)

print("Results saved:", model_significance_path)
print("Validation saved:", section_14_validation_path)
print("Summary saved:", section_14_summary_path)

print("=" * 75)

assert all(section_14_checks.values()), (
    "Section 14 validation failed."
)

NOTEBOOK 19 — SECTION 14: MODEL-LEVEL SIGNIFICANCE
Model-level metric records: 3
Tests executed: 0
Tests unavailable: 3
Significant tests: 0
---------------------------------------------------------------------------
Validation passed: 6
Validation failed: 0
Section 14 status: PASS
Results saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/tables/model_level_significance.csv
Validation saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/validation/section_14_validation.csv
Summary saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/metadata/section_14_summary.json


In [49]:
# ============================================================
# SECTION 15 — SAVE STATISTICAL-TEST RESULTS
# ============================================================

OUTPUT_TABLES = {

    "pairwise_comparison_coverage.csv":
        pairwise_coverage_df,

    "wilcoxon_signed_rank_results.csv":
        wilcoxon_df,

    "friedman_test_results.csv":
        friedman_df,

    "posthoc_pairwise_results.csv":
        posthoc_df,

    "paired_effect_sizes.csv":
        effect_size_df,

    "paired_confidence_intervals.csv":
        confidence_interval_df,

    "significance_matrix.csv":
        significance_matrix_df,

    "significance_matrix_wide.csv":
        significance_matrix_wide_df,

    "dataset_level_significance.csv":
        dataset_significance_df,

    "model_level_significance.csv":
        model_significance_df,

    "missing_experimental_conditions.csv":
        missing_experiments_df

}

# ------------------------------------------------------------
# 1. SAVE INDIVIDUAL RESULT TABLES
# ------------------------------------------------------------

saved_table_records = []

for filename, dataframe in OUTPUT_TABLES.items():

    if not isinstance(dataframe, pd.DataFrame):
        raise TypeError(
            f"{filename} is not a pandas DataFrame."
        )

    output_path = TABLES_DIR / filename

    dataframe.to_csv(
        output_path,
        index=False
    )

    saved_table_records.append({

        "filename": filename,

        "rows": len(dataframe),

        "columns": len(dataframe.columns),

        "file_exists": output_path.is_file(),

        "file_size_bytes": (
            output_path.stat().st_size
            if output_path.is_file()
            else 0
        )

    })

saved_tables_manifest_df = pd.DataFrame(
    saved_table_records
)

# ------------------------------------------------------------
# 2. PREPARE CONSOLIDATED POST-HOC RESULTS
# ------------------------------------------------------------

consolidated_results = posthoc_df.copy()

merge_keys = [
    "dataset",
    "metric",
    "variant_a",
    "variant_b",
    "n_pairs"
]

# ------------------------------------------------------------
# 3. VALIDATE EFFECT-SIZE MERGE
# ------------------------------------------------------------

if not consolidated_results.empty:

    for name, dataframe in [
        ("effect_size_df", effect_size_df),
        ("confidence_interval_df", confidence_interval_df)
    ]:

        missing_keys = [
            key for key in merge_keys
            if key not in dataframe.columns
        ]

        if missing_keys:
            raise ValueError(
                f"{name} missing merge keys: {missing_keys}"
            )

    if effect_size_df.duplicated(merge_keys).any():
        raise ValueError(
            "Duplicate effect-size merge keys detected."
        )

    if confidence_interval_df.duplicated(merge_keys).any():
        raise ValueError(
            "Duplicate confidence-interval merge keys detected."
        )

    original_rows = len(consolidated_results)

    consolidated_results = consolidated_results.merge(

        effect_size_df,

        on=merge_keys,

        how="left",

        suffixes=("", "_effect"),

        validate="one_to_one"

    )

    if len(consolidated_results) != original_rows:
        raise AssertionError(
            "Effect-size merge changed the post-hoc row count."
        )

    consolidated_results = consolidated_results.merge(

        confidence_interval_df,

        on=merge_keys,

        how="left",

        suffixes=("", "_ci"),

        validate="one_to_one"

    )

    if len(consolidated_results) != original_rows:
        raise AssertionError(
            "Confidence-interval merge changed the post-hoc row count."
        )

# ------------------------------------------------------------
# 4. SAVE CONSOLIDATED RESULTS
# ------------------------------------------------------------

consolidated_path = (
    RESULTS_DIR / "consolidated_statistical_results.csv"
)

consolidated_results.to_csv(
    consolidated_path,
    index=False
)

# ------------------------------------------------------------
# 5. SAVE EXPORT MANIFEST
# ------------------------------------------------------------

manifest_path = (
    MANIFESTS_DIR / "section_15_export_manifest.csv"
)

saved_tables_manifest_df.to_csv(
    manifest_path,
    index=False
)

# ------------------------------------------------------------
# 6. VALIDATE SECTION 15
# ------------------------------------------------------------

section_15_checks = {

    "all_expected_tables_registered": (
        len(OUTPUT_TABLES) == 11
    ),

    "all_individual_tables_saved": (
        saved_tables_manifest_df["file_exists"].all()
    ),

    "all_table_files_nonempty": (
        (
            saved_tables_manifest_df["file_size_bytes"] > 0
        ).all()
    ),

    "consolidated_file_saved": (
        consolidated_path.is_file()
    ),

    "consolidated_row_count_preserved": (
        len(consolidated_results) == len(posthoc_df)
    ),

    "export_manifest_saved": (
        manifest_path.is_file()
    ),

}

section_15_validation_df = pd.DataFrame([
    {
        "check": name,
        "passed": passed
    }
    for name, passed in section_15_checks.items()
])

section_15_validation_path = (
    VALIDATION_DIR / "section_15_validation.csv"
)

section_15_validation_df.to_csv(
    section_15_validation_path,
    index=False
)

# ------------------------------------------------------------
# 7. SAVE SECTION SUMMARY
# ------------------------------------------------------------

section_15_summary = {

    "section": "15",

    "status": (
        "PASS"
        if all(section_15_checks.values())
        else "FAIL"
    ),

    "individual_tables_saved": len(OUTPUT_TABLES),

    "consolidated_rows": len(consolidated_results),

    "posthoc_source_rows": len(posthoc_df),

    "export_manifest_path": str(manifest_path),

    "consolidated_results_path": str(consolidated_path),

    "validation_path": str(section_15_validation_path),

    "note": (
        "Statistical result tables preserve unavailable "
        "measurements as missing values. No missing inferential "
        "evidence is interpreted as a non-significant result."
    )

}

section_15_summary_path = (
    METADATA_DIR / "section_15_summary.json"
)

with open(
    section_15_summary_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        section_15_summary,
        file,
        indent=4,
        default=str
    )

# ------------------------------------------------------------
# 8. DISPLAY RESULTS
# ------------------------------------------------------------

print("=" * 75)
print("NOTEBOOK 19 — SECTION 15: SAVE STATISTICAL-TEST RESULTS")
print("=" * 75)

print(
    "Individual result tables:",
    len(OUTPUT_TABLES)
)

print(
    "Consolidated result rows:",
    len(consolidated_results)
)

print(
    "Post-hoc source rows:",
    len(posthoc_df)
)

print("-" * 75)

print(
    "Validation passed:",
    int(section_15_validation_df["passed"].sum())
)

print(
    "Validation failed:",
    int((~section_15_validation_df["passed"]).sum())
)

print(
    "Section 15 status:",
    section_15_summary["status"]
)

print("Consolidated results:", consolidated_path)
print("Export manifest:", manifest_path)
print("Validation:", section_15_validation_path)
print("Summary:", section_15_summary_path)

print("=" * 75)

assert all(section_15_checks.values()), (
    "Section 15 validation failed."
)

NOTEBOOK 19 — SECTION 15: SAVE STATISTICAL-TEST RESULTS
Individual result tables: 11
Consolidated result rows: 54
Post-hoc source rows: 54
---------------------------------------------------------------------------
Validation passed: 6
Validation failed: 0
Section 15 status: PASS
Consolidated results: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/results/consolidated_statistical_results.csv
Export manifest: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/manifests/section_15_export_manifest.csv
Validation: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/validation/section_15_validation.csv
Summary: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/metadata/section_15_summary.json


In [56]:
# ============================================================
# SECTION 16 — ALIGN EXPECTED PAIRWISE COVERAGE
# ============================================================

EXPECTED_PAIRWISE_COMPARISONS = (
    len(DATASETS)
    * len(available_metrics)
    * len(list(itertools.combinations(VARIANTS, 2)))
)

EXPECTED_FRIEDMAN_TESTS = (
    len(DATASETS)
    * len(available_metrics)
)

print("Datasets:", len(DATASETS))
print("Available metrics:", available_metrics)
print("Number of available metrics:", len(available_metrics))
print("Variants:", len(VARIANTS))
print(
    "Unique variant pairs:",
    len(list(itertools.combinations(VARIANTS, 2)))
)
print(
    "Expected pairwise comparisons:",
    EXPECTED_PAIRWISE_COMPARISONS
)
print(
    "Expected Friedman tests:",
    EXPECTED_FRIEDMAN_TESTS
)

assert EXPECTED_PAIRWISE_COMPARISONS == 54
assert EXPECTED_FRIEDMAN_TESTS == 9

Datasets: 3
Available metrics: ['fidelity_score', 'utility_score', 'privacy_score']
Number of available metrics: 3
Variants: 4
Unique variant pairs: 6
Expected pairwise comparisons: 54
Expected Friedman tests: 9


In [59]:

# ============================================================
# SECTION 16 — VALIDATE TEST COVERAGE
# ============================================================

# ------------------------------------------------------------
# 1. EXPECTED TEST COVERAGE
# ------------------------------------------------------------

EXPECTED_PAIRWISE_COMPARISONS = (
    len(DATASETS)
    * len(available_metrics)
    * len(list(itertools.combinations(VARIANTS, 2)))
)

EXPECTED_FRIEDMAN_TESTS = (
    len(DATASETS) * len(available_metrics)
)

# ------------------------------------------------------------
# 2. OBSERVED TEST COVERAGE
# ------------------------------------------------------------

valid_wilcoxon_tests = (
    int(wilcoxon_df["status"].eq("TESTED").sum())
    if not wilcoxon_df.empty else 0
)

valid_friedman_tests = (
    int(friedman_df["status"].eq("TESTED").sum())
    if not friedman_df.empty else 0
)

valid_posthoc_tests = (
    int(
        (
            posthoc_df["status"].eq("TESTED")
            & posthoc_df["p_value"].notna()
        ).sum()
    )
    if not posthoc_df.empty else 0
)

valid_effect_sizes = (
    int(
        effect_size_df[
            "rank_biserial_correlation"
        ].notna().sum()
    )
    if not effect_size_df.empty else 0
)

valid_confidence_intervals = (
    int(
        confidence_interval_df["ci_lower"].notna().sum()
    )
    if not confidence_interval_df.empty else 0
)

# ------------------------------------------------------------
# 3. CHECK STRUCTURAL ARTIFACTS
# ------------------------------------------------------------

expected_table_files = [
    TABLES_DIR / filename
    for filename in OUTPUT_TABLES
]

missing_table_files = [
    path.name
    for path in expected_table_files
    if not path.is_file()
]

saved_table_count = (
    len(expected_table_files) - len(missing_table_files)
)

structural_checks = [

    {
        "check": "input_source_loaded",
        "observed": SOURCE_STATUS,
        "passed": SOURCE_STATUS.startswith("LOADED")
    },

    {
        "check": "experimental_key_schema",
        "observed": required_keys_present,
        "passed": bool(required_keys_present)
    },

    {
        "check": "expected_experimental_grid",
        "observed": len(valid_unique_df),
        "expected": EXPECTED_RECORDS,
        "passed": len(valid_unique_df) == EXPECTED_RECORDS
    },

    {
        "check": "duplicate_experiments",
        "observed": len(duplicate_records),
        "expected": 0,
        "passed": len(duplicate_records) == 0
    },

    {
        "check": "pairwise_comparison_coverage",
        "observed": len(pairwise_coverage_df),
        "expected": EXPECTED_PAIRWISE_COMPARISONS,
        "passed": (
            len(pairwise_coverage_df)
            == EXPECTED_PAIRWISE_COMPARISONS
        )
    },

    {
        "check": "statistical_artifacts_saved",
        "observed": saved_table_count,
        "expected": len(OUTPUT_TABLES),
        "missing_files": "; ".join(missing_table_files),
        "passed": len(missing_table_files) == 0
    }

]

# ------------------------------------------------------------
# 4. EMPIRICAL EVIDENCE COVERAGE
# ------------------------------------------------------------

empirical_checks = [

    {
        "check": "wilcoxon_tests_available",
        "observed": valid_wilcoxon_tests,
        "expected": EXPECTED_PAIRWISE_COMPARISONS,
        "passed": valid_wilcoxon_tests > 0
    },

    {
        "check": "friedman_tests_available",
        "observed": valid_friedman_tests,
        "expected": EXPECTED_FRIEDMAN_TESTS,
        "passed": valid_friedman_tests > 0
    },

    {
        "check": "posthoc_tests_available",
        "observed": valid_posthoc_tests,
        "expected": EXPECTED_PAIRWISE_COMPARISONS,
        "passed": valid_posthoc_tests > 0
    },

    {
        "check": "effect_size_coverage",
        "observed": valid_effect_sizes,
        "expected": EXPECTED_PAIRWISE_COMPARISONS,
        "passed": valid_effect_sizes > 0
    },

    {
        "check": "confidence_interval_coverage",
        "observed": valid_confidence_intervals,
        "expected": EXPECTED_PAIRWISE_COMPARISONS,
        "passed": valid_confidence_intervals > 0
    }

]

# ------------------------------------------------------------
# 5. COMBINE COVERAGE RESULTS
# ------------------------------------------------------------

coverage_records = []

for record in structural_checks:

    coverage_records.append({
        **record,
        "check_type": "STRUCTURAL_VALIDATION",
        "status": (
            "PASS" if record["passed"] else "FAIL"
        )
    })

for record in empirical_checks:

    coverage_records.append({
        **record,
        "check_type": "EMPIRICAL_EVIDENCE_COVERAGE",
        "status": (
            "AVAILABLE"
            if record["passed"]
            else "MISSING_EVIDENCE"
        )
    })

test_coverage_df = pd.DataFrame(coverage_records)

# ------------------------------------------------------------
# 6. CALCULATE COVERAGE STATUS
# ------------------------------------------------------------

structural_passed = sum(
    record["passed"]
    for record in structural_checks
)

structural_failed = (
    len(structural_checks) - structural_passed
)

empirical_available = sum(
    record["passed"]
    for record in empirical_checks
)

empirical_missing = (
    len(empirical_checks) - empirical_available
)

coverage_status = (
    "PASS"
    if structural_failed == 0
    else "FAIL"
)

evidence_status = (
    "COMPLETE"
    if empirical_missing == 0
    else "MISSING_EMPIRICAL_EVIDENCE"
)

# ------------------------------------------------------------
# 7. SAVE COVERAGE RESULTS
# ------------------------------------------------------------

test_coverage_path = (
    VALIDATION_DIR / "statistical_test_coverage.csv"
)

test_coverage_df.to_csv(
    test_coverage_path,
    index=False
)

# ------------------------------------------------------------
# 8. VALIDATE SECTION 16
# ------------------------------------------------------------

section_16_checks = {

    "structural_validation_passed": (
        structural_failed == 0
    ),

    "coverage_table_created": (
        len(test_coverage_df)
        == len(structural_checks) + len(empirical_checks)
    ),

    "coverage_file_saved": (
        test_coverage_path.is_file()
    ),

    "missing_evidence_classified": (
        test_coverage_df.loc[
            test_coverage_df["check_type"]
            == "EMPIRICAL_EVIDENCE_COVERAGE",
            "status"
        ].isin(
            ["AVAILABLE", "MISSING_EVIDENCE"]
        ).all()
    ),

    "empirical_evidence_status_consistent": (
        (
            evidence_status == "COMPLETE"
            and empirical_missing == 0
        )
        or
        (
            evidence_status == "MISSING_EMPIRICAL_EVIDENCE"
            and empirical_missing > 0
        )
    ),

    "expected_metric_scope_consistent": (
        EXPECTED_PAIRWISE_COMPARISONS == (
            len(DATASETS)
            * len(available_metrics)
            * len(list(itertools.combinations(VARIANTS, 2)))
        )
        and
        EXPECTED_FRIEDMAN_TESTS == (
            len(DATASETS) * len(available_metrics)
        )
    )

}

section_16_validation_df = pd.DataFrame([
    {
        "check": name,
        "passed": passed
    }
    for name, passed in section_16_checks.items()
])

section_16_validation_path = (
    VALIDATION_DIR / "section_16_validation.csv"
)

section_16_validation_df.to_csv(
    section_16_validation_path,
    index=False
)

# ------------------------------------------------------------
# 9. SAVE SECTION SUMMARY
# ------------------------------------------------------------

section_16_summary = {

    "section": "16",

    "status": (
        "PASS"
        if all(section_16_checks.values())
        else "FAIL"
    ),

    "coverage_status": coverage_status,

    "evidence_status": evidence_status,

    "datasets": DATASETS,

    "metrics": available_metrics,

    "variants": VARIANTS,

    "expected_pairwise_comparisons": (
        EXPECTED_PAIRWISE_COMPARISONS
    ),

    "expected_friedman_tests": (
        EXPECTED_FRIEDMAN_TESTS
    ),

    "wilcoxon_tests_available": valid_wilcoxon_tests,

    "friedman_tests_available": valid_friedman_tests,

    "posthoc_tests_available": valid_posthoc_tests,

    "valid_effect_sizes": valid_effect_sizes,

    "valid_confidence_intervals": (
        valid_confidence_intervals
    ),

    "structural_checks_passed": structural_passed,

    "structural_checks_failed": structural_failed,

    "empirical_evidence_checks_available": (
        empirical_available
    ),

    "empirical_evidence_checks_missing": (
        empirical_missing
    ),

    "missing_artifact_files": missing_table_files,

    "interpretation": (
        "Pairwise coverage uses unique unordered variant pairs "
        "across the three available primary metrics. Structural "
        "coverage is validated independently from empirical "
        "test availability. Missing measurements are not "
        "interpreted as non-significant findings."
    ),

    "coverage_path": str(test_coverage_path),

    "validation_path": str(section_16_validation_path)

}

section_16_summary_path = (
    METADATA_DIR / "section_16_summary.json"
)

with open(
    section_16_summary_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        section_16_summary,
        file,
        indent=4,
        default=str
    )

# ------------------------------------------------------------
# 10. DISPLAY RESULTS
# ------------------------------------------------------------

print("=" * 75)
print("NOTEBOOK 19 — SECTION 16: VALIDATE TEST COVERAGE")
print("=" * 75)

print("Datasets:", len(DATASETS))
print("Available metrics:", len(available_metrics))
print("Variants:", len(VARIANTS))

print(
    "Expected pairwise comparisons:",
    EXPECTED_PAIRWISE_COMPARISONS
)

print(
    "Expected Friedman tests:",
    EXPECTED_FRIEDMAN_TESTS
)

print("Wilcoxon tests available:", valid_wilcoxon_tests)
print("Friedman tests available:", valid_friedman_tests)
print("Post-hoc tests available:", valid_posthoc_tests)
print("Valid effect sizes:", valid_effect_sizes)
print("Valid confidence intervals:", valid_confidence_intervals)

print("-" * 75)

print(
    "Structural checks passed:",
    structural_passed,
    "/",
    len(structural_checks)
)

print("Structural checks failed:", structural_failed)

print(
    "Empirical evidence checks available:",
    empirical_available,
    "/",
    len(empirical_checks)
)

print("Empirical evidence checks missing:", empirical_missing)

print("Missing artifact files:", missing_table_files)

print("Coverage status:", coverage_status)
print("Evidence status:", evidence_status)

print("-" * 75)

print(
    "Section validation passed:",
    int(section_16_validation_df["passed"].sum())
)

print(
    "Section validation failed:",
    int((~section_16_validation_df["passed"]).sum())
)

print("Section 16 status:", section_16_summary["status"])

print("Coverage saved:", test_coverage_path)
print("Validation saved:", section_16_validation_path)
print("Summary saved:", section_16_summary_path)

print("=" * 75)

assert all(section_16_checks.values()), (
    "Section 16 structural validation failed."
)


NOTEBOOK 19 — SECTION 16: VALIDATE TEST COVERAGE
Datasets: 3
Available metrics: 3
Variants: 4
Expected pairwise comparisons: 54
Expected Friedman tests: 9
Wilcoxon tests available: 0
Friedman tests available: 0
Post-hoc tests available: 0
Valid effect sizes: 0
Valid confidence intervals: 0
---------------------------------------------------------------------------
Structural checks passed: 6 / 6
Structural checks failed: 0
Empirical evidence checks available: 0 / 5
Empirical evidence checks missing: 5
Missing artifact files: []
Coverage status: PASS
Evidence status: MISSING_EMPIRICAL_EVIDENCE
---------------------------------------------------------------------------
Section validation passed: 6
Section validation failed: 0
Section 16 status: PASS
Coverage saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/validation/statistical_test_coverage.csv
Validation saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19/validation/section_16_valid

In [61]:

# ============================================================
# SECTION 17 — COMPLETION SUMMARY
# ============================================================

# ------------------------------------------------------------
# 1. EXECUTION STATUS
# ------------------------------------------------------------

statistical_artifacts_saved = all(
    (TABLES_DIR / filename).is_file()
    for filename in OUTPUT_TABLES
) and consolidated_path.is_file()

execution_complete = bool(

    SOURCE_STATUS.startswith("LOADED")

    and required_keys_present

    and len(valid_unique_df) == EXPECTED_RECORDS

    and len(duplicate_records) == 0

    and statistical_artifacts_saved

    and test_coverage_path.is_file()

    and section_16_validation_path.is_file()

    and section_16_summary["status"] == "PASS"

)

# ------------------------------------------------------------
# 2. STRUCTURAL VALIDATION STATUS
# ------------------------------------------------------------

# Use Section 16's six structural validation checks.
# Do not count missing empirical measurements as failures.

structural_validation_passed = int(
    section_16_validation_df["passed"].sum()
)

structural_validation_failed = int(
    (~section_16_validation_df["passed"]).sum()
)

structural_validation_complete = bool(
    structural_validation_failed == 0
)

# Empirical evidence coverage is reported separately.

empirical_evidence_available = int(
    section_16_summary[
        "empirical_evidence_checks_available"
    ]
)

empirical_evidence_missing = int(
    section_16_summary[
        "empirical_evidence_checks_missing"
    ]
)

# ------------------------------------------------------------
# 3. RESEARCH TESTING COMPLETENESS
# ------------------------------------------------------------

research_testing_complete = bool(

    valid_wilcoxon_tests > 0

    and valid_friedman_tests > 0

    and valid_effect_sizes > 0

    and valid_confidence_intervals > 0

)

# ------------------------------------------------------------
# 4. FINAL NOTEBOOK STATUS
# ------------------------------------------------------------

if not execution_complete or not structural_validation_complete:

    FINAL_STATUS = "INCOMPLETE"

elif not research_testing_complete:

    FINAL_STATUS = "COMPLETED_WITH_MISSING_EVIDENCE"

else:

    FINAL_STATUS = "COMPLETED"

# ------------------------------------------------------------
# 5. PREPARE COMPLETION SUMMARY
# ------------------------------------------------------------

summary = {

    "notebook": NOTEBOOK_ID,

    "title": NOTEBOOK_NAME,

    "status": FINAL_STATUS,

    "execution_complete": execution_complete,

    "structural_validation_complete": (
        structural_validation_complete
    ),

    "research_testing_complete": (
        research_testing_complete
    ),

    "source_status": SOURCE_STATUS,

    "source_path": (
        str(SOURCE_PATH)
        if SOURCE_PATH is not None
        else None
    ),

    "expected_experiments": EXPECTED_RECORDS,

    "loaded_experiments": len(evaluation_df),

    "valid_unique_experiments": len(valid_unique_df),

    "missing_experiments": len(missing_experiments_df),

    "duplicate_experiments": len(duplicate_records),

    "available_metrics": available_metrics,

    "expected_pairwise_comparisons": (
        EXPECTED_PAIRWISE_COMPARISONS
    ),

    "expected_friedman_tests": (
        EXPECTED_FRIEDMAN_TESTS
    ),

    "wilcoxon_tests_available": valid_wilcoxon_tests,

    "friedman_tests_available": valid_friedman_tests,

    "posthoc_tests_available": valid_posthoc_tests,

    "valid_effect_sizes": valid_effect_sizes,

    "valid_confidence_intervals": (
        valid_confidence_intervals
    ),

    "significant_holm_comparisons": (

        int(
            posthoc_df["significant_holm"].fillna(False).sum()
        )
        if not posthoc_df.empty
        else 0

    ),

    "significant_friedman_tests": (

        int(
            friedman_df["significant_holm"].fillna(False).sum()
        )
        if not friedman_df.empty
        else 0

    ),

    "alpha": ALPHA,

    "confidence_level": CONFIDENCE_LEVEL,

    "multiple_comparison_method": (
        MULTIPLE_TEST_METHOD
    ),

    "structural_validation_passed": (
        structural_validation_passed
    ),

    "structural_validation_failed": (
        structural_validation_failed
    ),

    "empirical_evidence_checks_available": (
        empirical_evidence_available
    ),

    "empirical_evidence_checks_missing": (
        empirical_evidence_missing
    ),

    "evidence_status": (
        section_16_summary["evidence_status"]
    ),

    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat()

}

summary_path = (
    METADATA_DIR / "notebook_19_completion_summary.json"
)

# ------------------------------------------------------------
# 6. SAVE INITIAL COMPLETION SUMMARY
# ------------------------------------------------------------

with open(
    summary_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        summary,
        file,
        indent=4,
        default=str
    )

# ------------------------------------------------------------
# 7. GENERATE ARTIFACT INTEGRITY MANIFEST
# ------------------------------------------------------------

def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as file:

        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):

            digest.update(chunk)

    return digest.hexdigest()


integrity_path = (
    MANIFESTS_DIR / "artifact_integrity_manifest.csv"
)

integrity_records = []

# Exclude the manifest itself and the completion summary.
# The summary is written again after the manifest is created,
# so including it here would produce a stale hash.

for path in sorted(
    NOTEBOOK_ROOT.rglob("*")
):

    if not path.is_file():
        continue

    if path in [integrity_path, summary_path]:
        continue

    integrity_records.append({

        "relative_path": str(
            path.relative_to(NOTEBOOK_ROOT)
        ),

        "size_bytes": path.stat().st_size,

        "sha256": sha256_file(path)

    })


integrity_df = pd.DataFrame(

    integrity_records,

    columns=[
        "relative_path",
        "size_bytes",
        "sha256"
    ]

)

integrity_df.to_csv(
    integrity_path,
    index=False
)

# ------------------------------------------------------------
# 8. UPDATE FINAL SUMMARY
# ------------------------------------------------------------

summary["integrity_manifest"] = str(
    integrity_path
)

summary["integrity_manifest_files"] = len(
    integrity_df
)

summary["integrity_manifest_sha256"] = (
    sha256_file(integrity_path)
)

summary["completion_summary_sha256_excluded"] = True

# Save final summary only after the integrity manifest.
# Its hash is intentionally excluded from that manifest.

with open(
    summary_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        summary,
        file,
        indent=4,
        default=str
    )

# ------------------------------------------------------------
# 9. FINAL COMPLETION VALIDATION
# ------------------------------------------------------------

section_17_checks = {

    "execution_complete": execution_complete,

    "structural_validation_complete": (
        structural_validation_complete
    ),

    "completion_summary_saved": (
        summary_path.is_file()
    ),

    "integrity_manifest_saved": (
        integrity_path.is_file()
    ),

    "integrity_manifest_has_valid_hashes": (
        integrity_df["sha256"].str.fullmatch(
            r"[0-9a-f]{64}"
        ).all()
        if not integrity_df.empty
        else False
    ),

    "summary_and_manifest_consistent": (
        summary["integrity_manifest_sha256"]
        == sha256_file(integrity_path)
    ),

    "missing_evidence_reported_correctly": (
        (
            FINAL_STATUS == "COMPLETED_WITH_MISSING_EVIDENCE"
            and empirical_evidence_missing > 0
        )
        or
        (
            FINAL_STATUS == "COMPLETED"
            and research_testing_complete
        )
        or
        (
            FINAL_STATUS == "INCOMPLETE"
            and (
                not execution_complete
                or not structural_validation_complete
            )
        )
    )

}

section_17_validation_df = pd.DataFrame([
    {
        "check": name,
        "passed": passed
    }
    for name, passed in section_17_checks.items()
])

section_17_validation_path = (
    VALIDATION_DIR / "section_17_validation.csv"
)

section_17_validation_df.to_csv(
    section_17_validation_path,
    index=False
)

# ------------------------------------------------------------
# 10. MEMORY CLEANUP
# ------------------------------------------------------------

gc.collect()

# ------------------------------------------------------------
# 11. FINAL COMPLETION REPORT
# ------------------------------------------------------------

print("=" * 75)
print("NOTEBOOK 19 — COMPLETION SUMMARY")
print("=" * 75)

print("Status:", FINAL_STATUS)

print("Execution complete:", execution_complete)

print(
    "Structural validation complete:",
    structural_validation_complete
)

print(
    "Research testing complete:",
    research_testing_complete
)

print("-" * 75)

print("Expected experiments:", EXPECTED_RECORDS)

print("Loaded experiments:", len(evaluation_df))

print("Valid unique experiments:", len(valid_unique_df))

print("Missing experiments:", len(missing_experiments_df))

print("Duplicate experiments:", len(duplicate_records))

print("-" * 75)

print("Available metrics:", len(available_metrics))

print(
    "Expected pairwise comparisons:",
    EXPECTED_PAIRWISE_COMPARISONS
)

print("Wilcoxon tests available:", valid_wilcoxon_tests)

print("Friedman tests available:", valid_friedman_tests)

print("Post-hoc tests available:", valid_posthoc_tests)

print("Valid effect sizes:", valid_effect_sizes)

print(
    "Valid confidence intervals:",
    valid_confidence_intervals
)

print("-" * 75)

print(
    "Significant Holm pairwise:",
    summary["significant_holm_comparisons"]
)

print(
    "Significant Friedman tests:",
    summary["significant_friedman_tests"]
)

print("-" * 75)

print(
    "Structural validation passed:",
    structural_validation_passed
)

print(
    "Structural validation failed:",
    structural_validation_failed
)

print(
    "Empirical evidence available:",
    empirical_evidence_available
)

print(
    "Empirical evidence missing:",
    empirical_evidence_missing
)

print("-" * 75)

print(
    "Section 17 checks passed:",
    int(section_17_validation_df["passed"].sum())
)

print(
    "Section 17 checks failed:",
    int((~section_17_validation_df["passed"]).sum())
)

print("Integrity manifest files:", len(integrity_df))

print("-" * 75)

print("Results directory:", RESULTS_DIR)

print("Tables directory:", TABLES_DIR)

print("Figures directory:", FIGURES_DIR)

print("Completion summary:", summary_path)

print("Coverage report:", test_coverage_path)

print("Integrity manifest:", integrity_path)

print("Section 17 validation:", section_17_validation_path)

print("=" * 75)

assert all(section_17_checks.values()), (
    "Notebook 19 completion validation failed."
)


NOTEBOOK 19 — COMPLETION SUMMARY
Status: COMPLETED_WITH_MISSING_EVIDENCE
Execution complete: True
Structural validation complete: True
Research testing complete: False
---------------------------------------------------------------------------
Expected experiments: 60
Loaded experiments: 60
Valid unique experiments: 60
Missing experiments: 0
Duplicate experiments: 0
---------------------------------------------------------------------------
Available metrics: 3
Expected pairwise comparisons: 54
Wilcoxon tests available: 0
Friedman tests available: 0
Post-hoc tests available: 0
Valid effect sizes: 0
Valid confidence intervals: 0
---------------------------------------------------------------------------
Significant Holm pairwise: 0
Significant Friedman tests: 0
---------------------------------------------------------------------------
Structural validation passed: 6
Structural validation failed: 0
Empirical evidence available: 0
Empirical evidence missing: 5
---------------------------